In [ ]:
# ============================================================
# Reproduce Figure 3a,b (ERA5-based test version)
#   (a) vertical NAM index from ERA5 geopotential
#   (b) CPC AO index from text file
#
# Fixes:
# - Correct groupby dimension handling for climatology
# - Use dask with 8 workers / threads
# ============================================================

import os
import re
import glob
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import dask
from dask.diagnostics import ProgressBar

# -----------------------------
# Dask / parallel config
# -----------------------------
dask.config.set(scheduler="threads", num_workers=8)

# -----------------------------
# Paths
# -----------------------------
ERA5_Z_DIR = "/pool/datos/reanalisis/era5/6hourly/global"
AO_FILE    = "/mnt/soclim0/public_data/weiji/NOAA_AO_INEX_260228.txt"
OUT_DIR = "/home/weiji/restart_exam/code/20260315NAM/resul/debugLawrence"
SAVE_DATA_DIR = "/mnt/soclim0/public_data/weiji/era5/NAM"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(SAVE_DATA_DIR, exist_ok=True)

OUT_FIG = os.path.join(OUT_DIR, "Figure3ab_ERA5NAM_AOCPC_201911_202005.png")
OUT_NAM = os.path.join(SAVE_DATA_DIR, "ERA5_daily_vertical_NAM_20191101_20200531.nc")
OUT_AO  = os.path.join(SAVE_DATA_DIR, "AO_CPC_20191101_20200531.csv")
# -----------------------------
# Config
# -----------------------------
G = 9.80665
LAT_MIN = 65.0
LAT_MAX = 90.0
START_PLOT = "2019-11-01"
END_PLOT   = "2020-05-31"

YEAR_MIN = 1950
YEAR_MAX = 2020

# ============================================================
# 1) Find ERA5 pressure-level z files only (exclude ml)
# ============================================================
pattern = re.compile(r"^z_4daily_(\d{8})-(\d{8})\.nc$")

all_files = sorted(glob.glob(os.path.join(ERA5_Z_DIR, "z_4daily*.nc")))
z_files = []

for f in all_files:
    base = os.path.basename(f)
    m = pattern.match(base)
    if m is None:
        continue
    y0 = int(m.group(1)[:4])
    if YEAR_MIN <= y0 <= YEAR_MAX:
        z_files.append(f)

print("=" * 80)
print(f"Matched ERA5 pressure-level z files: {len(z_files)}")
print("First:", z_files[0])
print("Last: ", z_files[-1])
print("=" * 80)

# ============================================================
# 2) Open multi-file dataset
# ============================================================
ds = xr.open_mfdataset(
    z_files,
    combine="by_coords",
    parallel=True,
    chunks={
        "time": 180,
        "level": 31,
        "latitude": 73,
        "longitude": 144,
    }
)

print(ds)

# ============================================================
# 3) Convert geopotential -> geopotential height
# ============================================================
Z = ds["z"] / G
Z.name = "zg"
Z.attrs["long_name"] = "Geopotential height"
Z.attrs["units"] = "m"

# ============================================================
# 4) Daily mean
# ============================================================
Z_daily = Z.resample(time="1D").mean()

print("\nDaily mean Z:")
print(Z_daily)

# ============================================================
# 5) 65–90N polar-cap average
# ============================================================
# latitude is descending: 90 -> -90
Z_pc = Z_daily.sel(latitude=slice(LAT_MAX, LAT_MIN))

# mean over longitude
Z_pc_lonmean = Z_pc.mean("longitude")

# cosine-latitude weights
weights = np.cos(np.deg2rad(Z_pc_lonmean["latitude"]))

# weighted mean over latitude
Z_pc_daily = Z_pc_lonmean.weighted(weights).mean("latitude")
Z_pc_daily.name = "zg_polarcap"
Z_pc_daily.attrs["long_name"] = "65-90N polar-cap daily mean geopotential height"
Z_pc_daily.attrs["units"] = "m"

print("\nPolar-cap daily Z:")
print(Z_pc_daily)

# ============================================================
# 6) Build climatology excluding 2020
# ============================================================
ref = Z_pc_daily.sel(time=slice("1950-01-01", "2019-12-31"))

# Add month-day coordinate explicitly, so groupby name is stable
ref = ref.assign_coords(monthday=ref["time"].dt.strftime("%m-%d"))
clim = ref.groupby("monthday").mean("time")

# Daily anomalies over reference period
ref_anom = ref.groupby("monthday") - clim

# Standard deviation at each level from 1950-2019 anomalies
std = ref_anom.std("time")
std.name = "std_ref"

print("\nClimatology:")
print(clim)
print("\nReference std:")
print(std)

# ============================================================
# 7) Compute NAM for all available dates
#    NAM = - standardized polar-cap geopotential height anomaly
# ============================================================
all_data = Z_pc_daily.assign_coords(monthday=Z_pc_daily["time"].dt.strftime("%m-%d"))
anom_all = all_data.groupby("monthday") - clim
NAM_all = -(anom_all / std)
NAM_all.name = "NAM"
NAM_all.attrs["long_name"] = "Vertical NAM index from standardized 65-90N geopotential height anomalies"
NAM_all.attrs["description"] = (
    "Computed from ERA5 daily geopotential height over 65-90N, "
    "using month-day climatology from 1950-2019 (excluding 2020), "
    "then standardized by reference-period std and multiplied by -1."
)

print("\nNAM_all:")
print(NAM_all)

# ============================================================
# 8) Extract plot period
# ============================================================
NAM_plot = NAM_all.sel(time=slice(START_PLOT, END_PLOT))

# compute here to materialize before plotting/saving
with ProgressBar():
    NAM_plot = NAM_plot.load()

print("\nNAM_plot:")
print(NAM_plot)

# Save NAM
NAM_plot.to_dataset(name="NAM").to_netcdf(OUT_NAM)

# ============================================================
# 9) Read AO_CPC text (robust parser)
# ============================================================

rows = []

with open(AO_FILE, "r") as f:
    for line in f:

        parts = line.split()

        if len(parts) != 4:
            continue

        y, m, d, ao = parts

        try:
            y = int(y)
            m = int(m)
            d = int(d)
            ao = float(ao)
        except:
            continue

        # remove missing value
        if ao == -99.000:
            continue

        rows.append((y, m, d, ao))

ao = pd.DataFrame(rows, columns=["year","month","day","ao"])

ao["time"] = pd.to_datetime(
    dict(year=ao.year, month=ao.month, day=ao.day)
)

ao = ao[(ao["time"] >= START_PLOT) & (ao["time"] <= END_PLOT)].copy()

ao.to_csv(OUT_AO, index=False)
# ============================================================
# 10) Plot Figure 3a,b style
# ============================================================
fig = plt.figure(figsize=(10, 8))
gs = fig.add_gridspec(
    nrows=2, ncols=1,
    height_ratios=[3.0, 1.1],
    hspace=0.05
)

# -----------------------------
# Panel (a): NAM time-height
# -----------------------------
ax1 = fig.add_subplot(gs[0, 0])

levels = NAM_plot["level"].values
times = NAM_plot["time"].values
nam2d = NAM_plot.transpose("level", "time").values

clevs = np.arange(-4.25, 4.26, 0.5)
cf = ax1.contourf(
    times, levels, nam2d,
    levels=clevs,
    cmap="RdBu",
    extend="both"
)

ax1.set_yscale("log")
ax1.invert_yaxis()
ax1.set_ylim(1000, 1)
ax1.set_ylabel("Pressure [hPa]", fontsize=18)
ax1.set_yticks([1000, 300, 100, 30, 10, 3, 1])
ax1.get_yaxis().set_major_formatter(plt.ScalarFormatter())
ax1.tick_params(labelbottom=False)
ax1.text(0.01, 0.92, "(a)", transform=ax1.transAxes, fontsize=18, fontweight="bold")

cbar = fig.colorbar(cf, ax=ax1, pad=0.02)
cbar.set_label("NAM index", fontsize=16)

# -----------------------------
# Panel (b): AO_CPC
# -----------------------------
ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
ax2.plot(ao["time"], ao["ao"], lw=2)
ax2.axhline(0, color="k", lw=1)

ax2.set_ylabel(r"$AO_{\mathrm{CPC}}$", fontsize=18)
ax2.set_ylim(-4.5, 4.5)
ax2.set_yticks([-3, 0, 3])
ax2.text(0.01, 0.82, "(b)", transform=ax2.transAxes, fontsize=18, fontweight="bold")

for ax in [ax1, ax2]:
    ax.grid(True, alpha=0.3)

fig.savefig(OUT_FIG, dpi=200, bbox_inches="tight")
plt.show()

print("\nSaved:")
print("  Figure:", OUT_FIG)
print("  NAM   :", OUT_NAM)
print("  AO    :", OUT_AO)

In [ ]:
# ============================================================
# ERA5-based NOAA AO Index Validation Pipeline (Final Fixed Version)
# ============================================================

import os
import re
import glob
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import dask
from dask.diagnostics import ProgressBar
from eofs.xarray import Eof
from scipy.stats import pearsonr
import matplotlib.colors as mcolors

dask.config.set(scheduler="threads", num_workers=8)

# -----------------------------
# Paths (请确保与你服务器一致)
# -----------------------------
ERA5_Z_DIR = "/pool/datos/reanalisis/era5/6hourly/global"
AO_FILE    = "/mnt/soclim0/public_data/weiji/NOAA_AO_INEX_260228.txt"
OUT_DIR = "/home/weiji/restart_exam/code/20260315NAM/resul/debugLawrence"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_FIG_TS = os.path.join(OUT_DIR, "Validation_ERA5_vs_CPC_AO_Timeseries.png")
OUT_FIG_MAP = os.path.join(OUT_DIR, "Validation_ERA5_Regression_Map.png")

# -----------------------------
# Config
# -----------------------------
G = 9.80665
LAT_MIN = 20.0 
LAT_MAX = 90.0
START_PLOT = "2019-11-01"
END_PLOT   = "2020-05-31"
YEAR_MIN = 1950
YEAR_MAX = 2020

# 1) Find files
pattern = re.compile(r"^z_4daily_(\d{8})-(\d{8})\.nc$")
all_files = sorted(glob.glob(os.path.join(ERA5_Z_DIR, "z_4daily*.nc")))
z_files = [f for f in all_files if pattern.match(os.path.basename(f)) and YEAR_MIN <= int(pattern.match(os.path.basename(f)).group(1)[:4]) <= YEAR_MAX]

print(f"Matched ERA5 files: {len(z_files)}")

# 2) Open dataset and Auto-Detect Units
ds = xr.open_mfdataset(z_files, combine="by_coords", parallel=True, chunks={"time": 180})

print("⏳ 正在截取 1000 hPa 区域并加载到内存...")
with ProgressBar():
    Z_raw = ds["z"].sel(level=1000, latitude=slice(LAT_MAX, LAT_MIN)).compute()

# 【修复 1：智能判断是否需要除以重力加速度 G】
mean_val = Z_raw.mean().values
if mean_val > 500:
    print(f"🔧 检测到数据均值为 {mean_val:.1f}，判定为位势 (Geopotential)，正在除以 G...")
    Z_1000 = Z_raw / G
else:
    print(f"🔧 检测到数据均值为 {mean_val:.1f}，判定已经是位势高度 (m)，跳过除以 G！")
    Z_1000 = Z_raw

# 3) Daily mean
Z_daily = Z_1000.resample(time="1D").mean()

# ============================================================
# 4) NOAA Standard Phase 1: Monthly EOF on 1979-2000
# ============================================================
print("📊 计算 1979-2000 月平均气候态以提取经典 EOF...")
Z_base_monthly = Z_daily.sel(time=slice("1979-01-01", "2000-12-31")).resample(time="1MS").mean()
clim_monthly = Z_base_monthly.groupby("time.month").mean("time")
anom_monthly_base = Z_base_monthly.groupby("time.month") - clim_monthly

coslat = np.cos(np.deg2rad(anom_monthly_base.latitude))
wgts_2d = np.sqrt(coslat).broadcast_like(anom_monthly_base.isel(time=0))
solver = Eof(anom_monthly_base, weights=wgts_2d.values)

# 【修复 2：提取完全无缩放的原始 PC1，用于计算严谨的标准差】
pc1_raw = solver.pcs(npcs=1, pcscaling=0).squeeze() 
sigma_monthly = pc1_raw.std(dim='time') 

# 用于画图的回归场 (使用 pcscaling=1 以便显示 1个标准差 对应的高度变化 m)
eof1_reg = solver.eofsAsCovariance(neofs=1, pcscaling=1).squeeze()

if eof1_reg.sel(latitude=slice(90, 75)).mean() > 0:
    flip = -1
    eof1_reg = eof1_reg * -1
else:
    flip = 1

# ============================================================
# 5) NOAA Standard Phase 2: Daily Projection for all years
# ============================================================
print("📊 计算 1979-2000 日平均平滑气候态...")
Z_base_daily = Z_daily.sel(time=slice("1979-01-01", "2000-12-31"))
clim_daily = Z_base_daily.groupby("time.dayofyear").mean("time")
clim_daily_smooth = clim_daily.rolling(dayofyear=21, center=True).mean().bfill("dayofyear").ffill("dayofyear")

anom_daily_all = Z_daily.groupby("time.dayofyear") - clim_daily_smooth
anom_daily_all = anom_daily_all.drop_vars("dayofyear")

print("📈 进行每日数据投影...")
# 使用 eofscaling=0 进行最纯净的空间投影
pseudo_pcs_raw = solver.projectField(anom_daily_all, neofs=1, eofscaling=0).squeeze() * flip

# 【核心：严格除以 1979-2000 年的 *月均* 标准差】
era5_ao = pseudo_pcs_raw / sigma_monthly

df_era5 = pd.DataFrame({"time": pd.to_datetime(era5_ao.time.values), "ao_era5": era5_ao.values})

# ============================================================
# 6) Load CPC and Plot
# ============================================================
print("📖 读取 CPC 官方数据...")
rows = []
with open(AO_FILE, "r") as f:
    for line in f:
        parts = line.split()
        if len(parts) == 4 and float(parts[3]) != -99.000:
            rows.append((int(parts[0]), int(parts[1]), int(parts[2]), float(parts[3])))

df_cpc = pd.DataFrame(rows, columns=["year","month","day","ao_cpc"])
df_cpc["time"] = pd.to_datetime(dict(year=df_cpc.year, month=df_cpc.month, day=df_cpc.day))

df_merged = pd.merge(df_era5, df_cpc[['time', 'ao_cpc']], on='time', how='inner')
# 强制将字符串转换为 Pandas 时间戳进行精准过滤
df_plot = df_merged[(df_merged["time"] >= pd.to_datetime(START_PLOT)) & 
                        (df_merged["time"] <= pd.to_datetime(END_PLOT))].copy()

    # 加上这两行诊断代码，如果图还是白的，终端会直接告诉你缺了哪段数据！
if len(df_plot) == 0:
    print(f"\n🚨 警告：在 {START_PLOT} 到 {END_PLOT} 之间没有匹配到数据！")
    print(f"当前数据的实际时间范围是：{df_merged['time'].min().date()} 到 {df_merged['time'].max().date()}")

r_val, _ = pearsonr(df_merged['ao_era5'], df_merged['ao_cpc'])
print(f"🏆 全时段 (1950-2020) ERA5 与 CPC AO 的相关系数 R = {r_val:.4f}")

# ================= 图 1: 对比折线图 =================
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), gridspec_kw={'height_ratios': [2, 1]})
ax1.plot(df_plot["time"], df_plot["ao_cpc"], lw=2.5, color='black', label='NOAA CPC Official')
ax1.plot(df_plot["time"], df_plot["ao_era5"], lw=1.5, color='red', linestyle='--', label='ERA5 Re-calculated')
ax1.axhline(0, color="grey", lw=1)
ax1.set_ylabel("Standardized AO Index", fontsize=14)
ax1.set_title(f"AO Index Validation ({START_PLOT} to {END_PLOT})\nOverall Pearson R = {r_val:.3f}", fontsize=16, fontweight="bold")
ax1.legend(loc='upper right', fontsize=12)
ax1.grid(True, alpha=0.3)

ax2.scatter(df_plot["ao_cpc"], df_plot["ao_era5"], alpha=0.6, color='blue', edgecolors='k')
min_val = min(df_plot["ao_cpc"].min(), df_plot["ao_era5"].min())
max_val = max(df_plot["ao_cpc"].max(), df_plot["ao_era5"].max())
ax2.plot([min_val, max_val], [min_val, max_val], 'k--', lw=2, label='1:1 Perfect Match')
ax2.set_xlabel("NOAA CPC AO Index", fontsize=14)
ax2.set_ylabel("ERA5 Calculated AO Index", fontsize=14)
ax2.legend(loc='upper left', fontsize=12)
ax2.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_FIG_TS, dpi=300, bbox_inches="tight")

# ================= 图 2: NOAA 风格回归验证图 =================
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

fig2 = plt.figure(figsize=(8, 8))
ax_map = plt.axes(projection=ccrs.NorthPolarStereo())
ax_map.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())
ax_map.coastlines(linewidth=1)
ax_map.add_feature(cfeature.BORDERS, linestyle=':', alpha=0.5)

lon = eof1_reg.longitude
lat = eof1_reg.latitude
eof1_reg_cyclic, lon_cyclic = add_cyclic_point(eof1_reg.values, coord=lon)

# ---------------- 核心修改：NOAA 离散色阶 ----------------
# 精确定义 13 个颜色块，完美对齐 NOAA 的配色
noaa_colors = [
        "#2d1e8f", # < -40 (深蓝)
        "#4355c7", # -40 to -35
        "#5a8ce6", # -35 to -30
        "#7ebbf5", # -30 to -25
        "#a3dcfb", # -25 to -20
        "#c2f0fb", # -20 to -15
        "#dff8fd", # -15 to -10
        "#f0fcfd", # -10 to -5
        "#ffffff", # -5 to 5 (纯白区域！)
        "#fefbd9", # 5 to 10
        "#feea9e", # 10 to 15
        "#fec762", # 15 to 20
        "#f27932", # 20 to 25
        "#b82522"  # > 25 (深红)
    ]
    # 强制边界点（注意包含了 -5 和 5）
bounds = [-45, -40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25, 30]
    
cmap_noaa = mcolors.ListedColormap(noaa_colors)
norm_noaa = mcolors.BoundaryNorm(bounds, cmap_noaa.N)
    # ---------------------------------------------------------

    # 绘制时传入自定义的 cmap 和 norm
cf = ax_map.contourf(lon_cyclic, lat, eof1_reg_cyclic, levels=bounds, 
                         transform=ccrs.PlateCarree(), cmap=cmap_noaa, norm=norm_noaa, extend='both')
                         
    # Colorbar 刻度标签也要跳过 0，显示 -5 和 5
cbar_ticks = [-40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25]
cbar = plt.colorbar(cf, ax=ax_map, orientation='vertical', shrink=0.7, pad=0.05, ticks=cbar_ticks)
    
plt.title(f'ERA5 Validation EOF shown as\nregression map of 1000mb height (m)', fontsize=16, fontname='Comic Sans MS', pad=20)

fig2.savefig(OUT_FIG_MAP, dpi=300, bbox_inches="tight")
print(f"✅ 图表已保存！")

In [ ]:
import os
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ================= 1. 路径配置 =================
NAM_FILE = "/home/weiji/restart_exam/code/20260315NAM/resul/debugLawrence/ERA5_daily_vertical_NAM_20191101_20200531.nc"
AO_FILE  = "/home/weiji/restart_exam/code/20260315NAM/resul/debugLawrence/AO_CPC_20191101_20200531.csv"

FIG_OUT_DIR = "/home/weiji/restart_exam/code/20260315NAM/resul/debugLawrence"
os.makedirs(FIG_OUT_DIR, exist_ok=True)

OUT_FIG = os.path.join(FIG_OUT_DIR, "Figure3ab_ERA5NAM_AOCPC_201911_202005_replot.png")

START_DATE = "2019-11-01"
END_DATE   = "2020-05-31"

print("📂 正在读取已经保存好的 NAM 和 AO 数据...")

# ================= 2. 读取数据 =================
# ---- NAM ----
ds_nam = xr.open_dataset(NAM_FILE)
nam_da = ds_nam["NAM"]   # 你的变量名就是 NAM

# 再保险切一次时间
nam_sub = nam_da.sel(time=slice(START_DATE, END_DATE))

# ---- AO ----
ao_df = pd.read_csv(AO_FILE, parse_dates=["time"])
ao_sub = ao_df[(ao_df["time"] >= START_DATE) & (ao_df["time"] <= END_DATE)].copy()

# 提取数组
time_arr = nam_sub["time"].values
ao_time  = ao_sub["time"].values
ao_vals  = ao_sub["ao"].values

print("🎨 正在按新风格重绘 Figure 3a,b ...")

# ================= 3. 新风格绘图 =================
fig = plt.figure(figsize=(10, 6))

gs = fig.add_gridspec(
    nrows=2, ncols=2,
    height_ratios=[2.0, 1.0],
    width_ratios=[1, 0.02],
    hspace=0.08, wspace=0.02
)

# -------------------------------------------------
# Panel (a): NAM time-height
# -------------------------------------------------
ax1 = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[0, 1])

levels_c = np.arange(-4.5, 5.0, 0.5)

cf = nam_sub.plot.contourf(
    ax=ax1,
    x="time",
    y="level",            # 你的坐标名是 level
    levels=levels_c,
    cmap="RdBu",
    extend="both",
    add_colorbar=False
)

ax1.set_yscale("log")
ax1.invert_yaxis()
ax1.set_ylim(1000, 1)
ax1.set_yticks([1000, 300, 100, 30, 10, 3, 1])
ax1.get_yaxis().set_major_formatter(plt.ScalarFormatter())

ax1.set_ylabel("Pressure [hPa]", fontsize=16)
ax1.set_xlabel("")
ax1.tick_params(labelbottom=False)
ax1.set_title("")
ax1.text(
    0.01, 0.92, "(a)",
    transform=ax1.transAxes,
    fontsize=16, fontweight="bold",
    bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=2)
)

cbar = fig.colorbar(cf, cax=cax)
cbar.set_label("NAM Index", fontsize=14)

# -------------------------------------------------
# Panel (b): AO_CPC
# -------------------------------------------------
ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)

ax2.plot(ao_time, ao_vals, color="black", lw=1.8)
ax2.axhline(0, color="gray", lw=1.2, linestyle="-")

ax2.set_ylabel(r"$AO_{\mathrm{CPC}}$", fontsize=16)
ax2.set_ylim(-4.5, 4.5)
ax2.set_yticks([-3, 0, 3])
ax2.text(
    0.01, 0.82, "(b)",
    transform=ax2.transAxes,
    fontsize=16, fontweight="bold",
    bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=2)
)
ax2.grid(True, linestyle=":", alpha=0.6)

# ================= 4. 标题与保存 =================
fig.suptitle(
    "Time series of ERA5 vertical NAM and CPC AO indices (2019/11–2020/05)",
    fontsize=18, fontweight="bold", y=0.95
)

plt.savefig(OUT_FIG, dpi=300, bbox_inches="tight")
print(f"✅ 已保存到: {OUT_FIG}")

plt.show()
ds_nam.close()

In [ ]:
# ============================================================
# Compare:
#   A) Simplified daily NAM from code A (already computed)
#   B) EOF-based daily NAM:
#        EOF1 trained STRICTLY from Samuel-style MONTHLY EOF,
#        then projected onto DAILY anomalies
#
# Output:
#   - daily EOF-based NAM
#   - correlation profile with code A daily NAM
#   - debug csv
#   - correlation figure
# ============================================================

import os
import re
import glob
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import dask
from dask.diagnostics import ProgressBar

warnings.filterwarnings("ignore")

# -----------------------------
# Dask config
# -----------------------------
dask.config.set(scheduler="threads", num_workers=8)

# -----------------------------
# Paths
# -----------------------------
ERA5_Z_DIR = "/pool/datos/reanalisis/era5/6hourly/global"

# Precomputed simplified NAM from code A
A_NAM_FILE = "/mnt/soclim0/public_data/weiji/era5/NAM/ERA5_daily_vertical_NAM_20191101_20200531.nc"

OUT_DIR = "/home/weiji/restart_exam/code/20260315NAM/resul/compare_A_vs_monthlyEOF_dailyProjection"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_EOF_DAILY_NAM = os.path.join(
    OUT_DIR, "ERA5_daily_vertical_NAM_monthlyEOF_projectedToDaily_20191101_20200531.nc"
)
OUT_CORR_NC = os.path.join(
    OUT_DIR, "ERA5_NAM_A_vs_monthlyEOFdaily_corr_profile_20191101_20200531.nc"
)
OUT_CORR_CSV = os.path.join(
    OUT_DIR, "ERA5_NAM_A_vs_monthlyEOFdaily_corr_profile_20191101_20200531.csv"
)
OUT_DEBUG_CSV = os.path.join(
    OUT_DIR, "ERA5_monthlyEOF_debug_info_by_level.csv"
)
OUT_FIG = os.path.join(
    OUT_DIR, "ERA5_NAM_A_vs_monthlyEOFdaily_corr_profile_20191101_20200531.png"
)

# -----------------------------
# Config
# -----------------------------
G = 9.80665

YEAR_MIN = 1950
YEAR_MAX = 2020

REF_START = "1950-01-01"
REF_END   = "2019-12-31"

START_PLOT = "2019-11-01"
END_PLOT   = "2020-05-31"

LAT_MIN_EOF = 20.0
LAT_MAX_EOF = 90.0

# ============================================================
# Helper functions
# ============================================================
def preprocess(ds):
    return ds[["z"]].sel(latitude=slice(LAT_MAX_EOF, LAT_MIN_EOF))

def nan_corr(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return np.nan, int(m.sum())
    return np.corrcoef(x[m], y[m])[0, 1], int(m.sum())

def check_matrix(name, X):
    X = np.asarray(X)
    n_nan = np.isnan(X).sum()
    n_inf = np.isinf(X).sum()
    finite_ratio = np.isfinite(X).sum() / X.size
    print(f"    {name}: shape={X.shape}, nan={n_nan}, inf={n_inf}, finite_ratio={finite_ratio:.6f}")
    if np.isfinite(X).any():
        print(f"    {name}: min={np.nanmin(X):.6e}, max={np.nanmax(X):.6e}")

def monthly_climatology_anom(da, ref_start, ref_end):
    """
    da dims: time, level, latitude
    returns:
        clim_mon: dims month, level, latitude
        anom_all: dims time, level, latitude
    """
    ref = da.sel(time=slice(ref_start, ref_end))
    clim_mon = ref.groupby("time.month").mean("time")
    anom_all = da.groupby("time.month") - clim_mon
    return clim_mon, anom_all

def build_samuel_weights(lat_vals):
    """
    Samuel: sqrt(cos(lat))
    Need robust clipping because cos(90°) may become a tiny negative number
    due to floating-point precision.
    """
    lat64 = lat_vals.astype(np.float64)
    coslat = np.cos(np.deg2rad(lat64))
    coslat = np.clip(coslat, 0.0, None)
    weights = np.sqrt(coslat)

    print("Latitude / weight debug:")
    print(f"    lat dtype          = {lat_vals.dtype}")
    print(f"    weights finite     = {np.isfinite(weights).sum()} / {weights.size}")
    print(f"    coslat min/max     = {np.nanmin(coslat):.8e} / {np.nanmax(coslat):.8e}")
    print(f"    weights min/max    = {np.nanmin(weights):.8e} / {np.nanmax(weights):.8e}")
    print("    First 5 lat/weight:")
    for la, w in zip(lat_vals[:5], weights[:5]):
        print(f"        lat={la:6.2f}, weight={w:.8e}")
    print("    Last 5 lat/weight:")
    for la, w in zip(lat_vals[-5:], weights[-5:]):
        print(f"        lat={la:6.2f}, weight={w:.8e}")
    print()

    return weights

def solve_eof1_monthly(Xref_w, lat_vals, debug_prefix=""):
    """
    Xref_w: weighted monthly anomalies, shape (lat, time)
    returns:
        eof1
        pc_ref
        train_mean
        method_used
        n_valid
    """
    valid_t = np.all(np.isfinite(Xref_w), axis=0)
    print(f"    [{debug_prefix}] valid monthly samples inside solver = {valid_t.sum()} / {valid_t.size}")

    X = Xref_w[:, valid_t]

    if X.shape[1] < 3:
        raise ValueError(f"{debug_prefix} Too few valid monthly samples: {X.shape[1]}")

    # Save training mean before centering, and use the same one for projection
    train_mean = X.mean(axis=1, keepdims=True)
    Xc = X - train_mean

    print(f"    [{debug_prefix}] solver matrix shape = {Xc.shape}")
    print(f"    [{debug_prefix}] solver matrix finite ratio = {np.isfinite(Xc).sum()/Xc.size:.6f}")
    print(f"    [{debug_prefix}] solver matrix min/max = {np.nanmin(Xc):.6e} / {np.nanmax(Xc):.6e}")

    # First try SVD
    try:
        U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
        eof1 = U[:, 0]
        pc_ref = eof1 @ Xc
        method_used = "svd"
        print(f"    [{debug_prefix}] SVD succeeded. Leading singular value = {S[0]:.6e}")

    except np.linalg.LinAlgError as e:
        print(f"    [{debug_prefix}] [WARN] SVD failed: {repr(e)}")
        print(f"    [{debug_prefix}] [WARN] Falling back to covariance eigen decomposition")

        C = Xc @ Xc.T
        C = 0.5 * (C + C.T)

        eigvals, eigvecs = np.linalg.eigh(C)
        idx = np.argsort(eigvals)[::-1]

        eof1 = eigvecs[:, idx[0]]
        pc_ref = eof1 @ Xc
        method_used = "eigh_fallback"
        print(f"    [{debug_prefix}] Leading eigenvalue = {eigvals[idx[0]]:.6e}")

    return eof1, pc_ref, train_mean, method_used, int(valid_t.sum())

def enforce_samuel_sign(eof1, lat_vals):
    """
    Samuel sign convention:
    if EOF1 at 80N > 0, flip sign
    """
    i80 = int(np.abs(lat_vals - 80.0).argmin())
    if eof1[i80] > 0:
        eof1 = -eof1
    return eof1

# ============================================================
# 1) Load simplified NAM from code A
# ============================================================
print("=" * 80)
print("1) LOAD SIMPLIFIED NAM FROM CODE A")
print("=" * 80)

dsA = xr.open_dataset(A_NAM_FILE)
namA = dsA["NAM"].sel(time=slice(START_PLOT, END_PLOT)).transpose("time", "level")
print(namA)
print()

# ============================================================
# 2) Find ERA5 z files only
# ============================================================
print("=" * 80)
print("2) FIND ERA5 z FILES")
print("=" * 80)

pattern = re.compile(r"^z_4daily_(\d{8})-(\d{8})\.nc$")
all_files = sorted(glob.glob(os.path.join(ERA5_Z_DIR, "z_4daily*.nc")))

z_files = []
for f in all_files:
    base = os.path.basename(f)
    m = pattern.match(base)
    if m is None:
        continue
    y0 = int(m.group(1)[:4])
    if YEAR_MIN <= y0 <= YEAR_MAX:
        z_files.append(f)

print(f"Matched ERA5 pressure-level z files: {len(z_files)}")
print("First:", z_files[0])
print("Last: ", z_files[-1])
print()

# ============================================================
# 3) Open ERA5 and build daily zonal-mean geopotential height
# ============================================================
print("=" * 80)
print("3) OPEN ERA5 AND BUILD DAILY ZONAL-MEAN ZG")
print("=" * 80)

ds = xr.open_mfdataset(
    z_files,
    combine="by_coords",
    parallel=True,
    preprocess=preprocess,
    chunks={
        "time": 180,
        "level": 31,
        "latitude": 29,
        "longitude": 144,
    }
)

zg_4x_zm = (ds["z"] / G).mean("longitude")
zg_4x_zm.name = "zg_zm_4x"
zg_4x_zm.attrs["units"] = "m"

with ProgressBar():
    zg_daily_zm = zg_4x_zm.resample(time="1D").mean().astype("float32").load()

ds.close()

print(zg_daily_zm)
print()

# ============================================================
# 4) Build monthly EOF training data (strict Samuel style)
# ============================================================
print("=" * 80)
print("4) BUILD MONTHLY EOF TRAINING DATA (STRICT SAMUEL STYLE)")
print("=" * 80)

# Monthly mean zonal-mean ZG
zg_monthly_zm = zg_daily_zm.resample(time="MS").mean()

# Monthly climatology and monthly anomalies
clim_mon, anom_monthly_all = monthly_climatology_anom(
    zg_monthly_zm, REF_START, REF_END
)

# Reference monthly anomalies for training EOF
anom_monthly_ref = anom_monthly_all.sel(time=slice(REF_START, REF_END)).transpose("time", "level", "latitude")

print("zg_monthly_zm shape  :", zg_monthly_zm.shape)
print("anom_monthly_ref shape:", anom_monthly_ref.shape)
print("Monthly reference span:", str(anom_monthly_ref["time"].values[0]), "to", str(anom_monthly_ref["time"].values[-1]))
print()

# ============================================================
# 5) Build daily anomalies for projection
#    IMPORTANT FIX:
#    Use 21-day smoothed DOY climatology to avoid cross-month jumps!
# ============================================================
print("=" * 80)
print("5) BUILD DAILY ANOMALIES FOR PROJECTION (DOY SMOOTHED)")
print("=" * 80)

# 计算每日参考期的 DOY 气候态并进行 21 天滑动平滑
ref_daily = zg_daily_zm.sel(time=slice(REF_START, REF_END))
clim_doy = ref_daily.groupby("time.dayofyear").mean("time")
clim_doy_smooth = (
    clim_doy
    .rolling(dayofyear=21, center=True)
    .mean()
    .bfill("dayofyear")
    .ffill("dayofyear")
)

# 计算基于 DOY 平滑气候态的每日异常场
anom_daily_all = zg_daily_zm.groupby("time.dayofyear") - clim_doy_smooth
if "dayofyear" in anom_daily_all.coords:
    anom_daily_all = anom_daily_all.drop_vars("dayofyear")

anom_daily_plot = anom_daily_all.sel(time=slice(START_PLOT, END_PLOT)).transpose("time", "level", "latitude")

# ============================================================
# 6) Compute monthly EOF1, then project to daily
# ============================================================
print("=" * 80)
print("6) COMPUTE MONTHLY EOF1, THEN PROJECT TO DAILY")
print("=" * 80)

levels = anom_monthly_ref["level"].values
lat_vals = anom_monthly_ref["latitude"].values
time_daily_plot = anom_daily_plot["time"].values

weights = build_samuel_weights(lat_vals)

# Convert to numpy
# monthly ref: (time, level, latitude) -> (level, latitude, time)
Xref_month = anom_monthly_ref.transpose("level", "latitude", "time").values.astype(np.float64)

# daily plot: (time, level, latitude) -> (level, latitude, time)
Xplot_day = anom_daily_plot.transpose("level", "latitude", "time").values.astype(np.float64)

nlev = len(levels)
ntime_plot = len(time_daily_plot)

eof_daily_nam = np.full((ntime_plot, nlev), np.nan, dtype=np.float64)
debug_rows = []

for ilev, lev in enumerate(levels):
    print("-" * 80)
    print(f"Level {ilev+1:02d}/{nlev:02d} : {lev} hPa")

    Xref = Xref_month[ilev, :, :]   # (lat, monthly_time)
    Xday = Xplot_day[ilev, :, :]    # (lat, daily_time)

    check_matrix("Xref_month_raw", Xref)
    check_matrix("Xday_raw", Xday)

    # Samuel weighting
    Xref_w = Xref * weights[:, None]
    check_matrix("Xref_month_weighted", Xref_w)

    valid_ref_t = np.all(np.isfinite(Xref_w), axis=0)
    print(f"    valid monthly samples after weighting = {valid_ref_t.sum()} / {valid_ref_t.size}")

    try:
        eof1, pc_ref, train_mean, method_used, n_month_ref = solve_eof1_monthly(
            Xref_w, lat_vals, debug_prefix=f"lev={lev}"
        )
    except Exception as e:
        print(f"    [ERROR] EOF solve failed at {lev} hPa: {repr(e)}")
        debug_rows.append({
            "level_hPa": lev,
            "status": "EOF_FAILED",
            "method": "none",
            "n_month_ref": int(valid_ref_t.sum()),
            "pc_ref_mean": np.nan,
            "pc_ref_std": np.nan,
            "corr_with_A": np.nan,
            "note": repr(e),
        })
        continue

    # Samuel sign convention
    eof1 = enforce_samuel_sign(eof1, lat_vals)

    # Recompute reference PC after sign fix, using SAME training mean
    Xref_w_valid = Xref_w[:, valid_ref_t]
    Xref_c = Xref_w_valid - train_mean

    pc_ref = eof1 @ Xref_c
    pc_ref_mean = np.mean(pc_ref)
    pc_ref_std = np.std(pc_ref, ddof=1)

    i80 = int(np.abs(lat_vals - 80.0).argmin())

    print(f"    method_used     = {method_used}")
    print(f"    n_month_ref     = {n_month_ref}")
    print(f"    pc_ref_mean     = {pc_ref_mean:.6e}")
    print(f"    pc_ref_std      = {pc_ref_std:.6e}")
    print(f"    eof1(near 80N)  = {eof1[i80]:.6e}")

    if (not np.isfinite(pc_ref_std)) or (pc_ref_std == 0):
        print("    [ERROR] pc_ref_std invalid; skip this level")
        debug_rows.append({
            "level_hPa": lev,
            "status": "STD_INVALID",
            "method": method_used,
            "n_month_ref": n_month_ref,
            "pc_ref_mean": pc_ref_mean,
            "pc_ref_std": pc_ref_std,
            "corr_with_A": np.nan,
            "note": "pc_ref_std invalid",
        })
        continue

    # --------------------------------------------------------
    # Project DAILY anomalies onto MONTHLY EOF1
    # --------------------------------------------------------
    valid_day_t = np.all(np.isfinite(Xday), axis=0)
    print(f"    valid daily samples for projection = {valid_day_t.sum()} / {valid_day_t.size}")

    nam_day_lev = np.full(Xday.shape[1], np.nan, dtype=np.float64)

    if valid_day_t.sum() > 0:
        Xday_valid = Xday[:, valid_day_t] * weights[:, None]
        Xday_c = Xday_valid - train_mean

        check_matrix("Xday_weighted_centered", Xday_c)

        pc_day = eof1 @ Xday_c
        nam_day_valid = (pc_day - pc_ref_mean) / pc_ref_std
        nam_day_lev[valid_day_t] = nam_day_valid

        print(f"    projected daily NAM min/max = {np.nanmin(nam_day_valid):.6e} / {np.nanmax(nam_day_valid):.6e}")

    eof_daily_nam[:, ilev] = nam_day_lev

    debug_rows.append({
        "level_hPa": lev,
        "status": "OK",
        "method": method_used,
        "n_month_ref": n_month_ref,
        "pc_ref_mean": pc_ref_mean,
        "pc_ref_std": pc_ref_std,
        "corr_with_A": np.nan,
        "note": "",
    })

print("-" * 80)
print()

# Build daily EOF NAM DataArray
namEOF_daily = xr.DataArray(
    eof_daily_nam,
    coords={"time": time_daily_plot, "level": levels},
    dims=("time", "level"),
    name="NAM_EOF_daily"
)

namEOF_daily.attrs["long_name"] = "Daily EOF-based NAM from monthly EOF projected onto daily anomalies"
namEOF_daily.attrs["description"] = (
    "EOF1 is trained strictly using Samuel's monthly EOF method: monthly mean "
    "zonal-mean geopotential height, monthly climatology removed, 20-90N, "
    "sqrt(cos(lat)) weighting, per pressure level, reference period 1950-2019. "
    "The trained EOF1 is then projected onto daily anomalies relative to monthly "
    "climatology, producing a daily EOF-based NAM index."
)
namEOF_daily.attrs["units"] = "1"

namEOF_daily.to_dataset(name="NAM_EOF_daily").to_netcdf(OUT_EOF_DAILY_NAM)

print("Saved EOF daily NAM:")
print(" ", OUT_EOF_DAILY_NAM)
print()

# ============================================================
# 7) Correlation with code A daily NAM
# ============================================================
print("=" * 80)
print("7) CORRELATION PROFILE WITH CODE A")
print("=" * 80)

common_times = np.intersect1d(namA["time"].values, namEOF_daily["time"].values)
common_levels = np.intersect1d(namA["level"].values, namEOF_daily["level"].values)

namA_cmp = namA.sel(time=common_times, level=common_levels).transpose("time", "level")
namEOF_cmp = namEOF_daily.sel(time=common_times, level=common_levels).transpose("time", "level")

corrs = []
nvalids = []

for lev in common_levels:
    x = namA_cmp.sel(level=lev).values
    y = namEOF_cmp.sel(level=lev).values

    r, nvalid = nan_corr(x, y)
    corrs.append(r)
    nvalids.append(nvalid)

    print(f"level={lev:>4} hPa : corr={r: .4f}, nvalid={nvalid}")

    for row in debug_rows:
        if row["level_hPa"] == lev:
            row["corr_with_A"] = r
            break

corr_da = xr.DataArray(
    np.array(corrs, dtype=np.float32),
    coords={"level": common_levels},
    dims=("level",),
    name="corr_A_vs_monthlyEOFdaily"
)

corr_da.attrs["long_name"] = "Correlation between code A NAM and monthly-EOF daily-projected NAM"
corr_da.attrs["units"] = "1"

nvalid_da = xr.DataArray(
    np.array(nvalids, dtype=np.int32),
    coords={"level": common_levels},
    dims=("level",),
    name="n_valid"
)

xr.Dataset({
    "corr_A_vs_monthlyEOFdaily": corr_da,
    "n_valid": nvalid_da
}).to_netcdf(OUT_CORR_NC)

pd.DataFrame({
    "level_hPa": common_levels,
    "corr_A_vs_monthlyEOFdaily": corrs,
    "n_valid": nvalids
}).to_csv(OUT_CORR_CSV, index=False)

pd.DataFrame(debug_rows).to_csv(OUT_DEBUG_CSV, index=False)

print()
print("Saved:")
print(" ", OUT_CORR_NC)
print(" ", OUT_CORR_CSV)
print(" ", OUT_DEBUG_CSV)
print()

# ============================================================
# 8) Plot
# ============================================================
print("=" * 80)
print("8) PLOT")
print("=" * 80)

fig, ax = plt.subplots(figsize=(6.3, 8.8))

ax.plot(
    corr_da.values,
    corr_da["level"].values,
    marker="o",
    lw=2
)

ax.axvline(0, color="k", lw=1, ls="--")

ax.set_yscale("log")
ax.set_ylim(1000, 1)
ax.set_yticks([1000, 300, 100, 30, 10, 3, 1])
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())

ax.set_xlim(-1.05, 1.05)
ax.set_xlabel("Correlation", fontsize=14)
ax.set_ylabel("Pressure [hPa]", fontsize=14)
ax.set_title(
    "Correlation profile:\nSimplified NAM (code A) vs monthly-EOF daily-projected NAM",
    fontsize=13
)

ax.grid(True, alpha=0.3)

fig.savefig(OUT_FIG, dpi=200, bbox_inches="tight")
plt.show()

print("Saved figure:")
print(" ", OUT_FIG)

In [ ]:
# ============================================================
# Compare AO from:
#   A) Code-A AO -> load if saved; otherwise recompute EXACTLY as Code A
#   B) STRICT Code-B AO method, but using the SAME ERA5 data as Code A
#
# Outputs:
#   1) AO_A (loaded or recomputed)
#   2) AO_B (computed from same ERA5, strict Code-B logic)
#   3) merged CSV on identical dates
#   4) full-period scatter plot: AO_A vs AO_B
#   5) short-window timeseries plot (2019-11 to 2020-06):
#        AO_A, AO_B, NOAA CPC
# ============================================================

import os
import re
import glob
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import dask

from dask.diagnostics import ProgressBar
from scipy.stats import pearsonr
from eofs.xarray import Eof as XarrayEof
from eofs.standard import Eof as StandardEof

dask.config.set(scheduler="threads", num_workers=8)

# ============================================================
# 0) PATHS & CONFIG
# ============================================================
ERA5_Z_DIR = "/pool/datos/reanalisis/era5/6hourly/global"
AO_FILE    = "/mnt/soclim0/public_data/weiji/NOAA_AO_INEX_260228.txt"

# 如果你有之前保存的代码A结果，可填真实路径；没有就设为 None
AO_A_PATH = None
# 例如：
# AO_A_PATH = "/home/weiji/restart_exam/code/20260315NAM/result/debugLawrence/ERA5_AO_from_codeA.nc"

OUT_DIR = "/home/weiji/restart_exam/code/20260324_AO_compare_A_vs_Bstyle_STRICT"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_A_AO_NC   = os.path.join(OUT_DIR, "AO_codeA.nc")
OUT_B_AO_NC   = os.path.join(OUT_DIR, "AO_codeBstyle_STRICT_from_same_ERA5.nc")
OUT_MERGEDCSV = os.path.join(OUT_DIR, "AO_compare_A_vs_Bstyle_STRICT_merged.csv")
OUT_SCATTER   = os.path.join(OUT_DIR, "AO_compare_A_vs_Bstyle_STRICT_scatter.png")
OUT_TS        = os.path.join(OUT_DIR, "AO_compare_201911_202006_timeseries_with_NOAA_STRICT.png")

G = 9.80665
LAT_MIN = 20.0
LAT_MAX = 90.0
YEAR_MIN = 1950
YEAR_MAX = 2020

# 这两个只用于“全时段 A vs B 合并/散点图”的过滤；一般保持 None
COMPARE_START = None
COMPARE_END   = None

# 这两个只用于折线图显示窗口
PLOT_START = "2019-11-01"
PLOT_END   = "2020-06-30"


# ============================================================
# 1) HELPERS
# ============================================================
def print_header(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def print_da_info(name, da):
    print(f"[{name}]")
    print(f"  type   : {type(da)}")
    print(f"  dims   : {da.dims}")
    print(f"  shape  : {da.shape}")
    if "time" in da.coords:
        try:
            print(f"  time   : {pd.to_datetime(da.time.values[0])} -> {pd.to_datetime(da.time.values[-1])}")
        except Exception:
            pass
    if "latitude" in da.coords:
        latv = da.latitude.values
        print(f"  lat    : {latv[0]} -> {latv[-1]}  (n={len(latv)})")
    if "longitude" in da.coords:
        lonv = da.longitude.values
        print(f"  lon    : {lonv[0]} -> {lonv[-1]}  (n={len(lonv)})")
    try:
        n_nan = int(np.isnan(da.values).sum())
        print(f"  NaN cnt: {n_nan}")
    except Exception:
        pass
    print()

def clean_time_series_da(da, new_name):
    da = da.squeeze(drop=True)

    extra_coord_names = []
    for c in da.coords:
        if c != "time" and c not in da.dims:
            extra_coord_names.append(c)
    if len(extra_coord_names) > 0:
        da = da.drop_vars(extra_coord_names)

    da = da.squeeze(drop=True)

    if da.ndim != 1 or da.dims != ("time",):
        raise ValueError(f"清理后仍不是 1D time series，当前 dims={da.dims}, shape={da.shape}")

    da.name = new_name
    return da

def maybe_filter_time_df(df, start=None, end=None):
    out = df.copy()
    if start is not None:
        out = out[out["time"] >= pd.to_datetime(start)]
    if end is not None:
        out = out[out["time"] <= pd.to_datetime(end)]
    return out.reset_index(drop=True)

def maybe_filter_time_da(da, start=None, end=None):
    out = da
    if start is not None and end is not None:
        out = out.sel(time=slice(pd.to_datetime(start), pd.to_datetime(end)))
    elif start is not None:
        out = out.sel(time=slice(pd.to_datetime(start), None))
    elif end is not None:
        out = out.sel(time=slice(None, pd.to_datetime(end)))
    return out

def find_era5_files(era5_dir, year_min=1950, year_max=2020):
    pattern = re.compile(r"^z_4daily_(\d{8})-(\d{8})\.nc$")
    all_files = sorted(glob.glob(os.path.join(era5_dir, "z_4daily*.nc")))

    z_files = []
    for f in all_files:
        base = os.path.basename(f)
        m = pattern.match(base)
        if m is None:
            continue
        y0 = int(m.group(1)[:4])
        if year_min <= y0 <= year_max:
            z_files.append(f)

    if len(z_files) == 0:
        raise FileNotFoundError("没有匹配到 ERA5 z_4daily 文件，请检查路径。")

    return z_files

def load_saved_ao(path):
    if path is None:
        raise FileNotFoundError("AO_A_PATH = None，视为不存在保存文件。")

    if not os.path.exists(path):
        raise FileNotFoundError(f"找不到 AO 文件: {path}")

    ext = os.path.splitext(path)[1].lower()

    if ext in [".nc", ".nc4", ".cdf"]:
        ds = xr.open_dataset(path)
        print("读取到的 nc 文件变量：", list(ds.data_vars))

        candidate_vars = ["ao_era5", "AO", "ao", "index", "pc1", "ao_A"]
        var_name = None
        for v in candidate_vars:
            if v in ds.data_vars:
                var_name = v
                break
        if var_name is None:
            if len(ds.data_vars) == 0:
                raise ValueError(f"{path} 中没有 data_vars。")
            var_name = list(ds.data_vars)[0]

        da = ds[var_name]
        da = clean_time_series_da(da, "ao_A")
        print_da_info("loaded ao_A", da)

        df = pd.DataFrame({
            "time": pd.to_datetime(da.time.values),
            "ao_A": da.values
        }).dropna().sort_values("time").reset_index(drop=True)
        return df

    elif ext in [".csv", ".txt"]:
        df = pd.read_csv(path, sep=None, engine="python")

        time_candidates = ["time", "date", "datetime", "Time", "Date"]
        time_col = None
        for c in time_candidates:
            if c in df.columns:
                time_col = c
                break
        if time_col is None:
            raise ValueError("保存文件里找不到时间列。")

        val_candidates = ["ao_era5", "AO", "ao", "index", "pc1", "ao_A"]
        val_col = None
        for c in val_candidates:
            if c in df.columns:
                val_col = c
                break
        if val_col is None:
            numeric_cols = [c for c in df.columns if c != time_col and pd.api.types.is_numeric_dtype(df[c])]
            if len(numeric_cols) == 0:
                raise ValueError("保存文件里找不到 AO 数值列。")
            val_col = numeric_cols[0]

        out = df[[time_col, val_col]].copy()
        out.columns = ["time", "ao_A"]
        out["time"] = pd.to_datetime(out["time"])
        out = out.dropna().sort_values("time").reset_index(drop=True)
        return out

    else:
        raise ValueError(f"暂不支持该文件格式: {ext}")

def load_noaa_ao_txt(ao_file):
    """
    按 code A 的方式读取 NOAA CPC AO 文本文件。
    跳过 value = -99.000 的坏值。
    返回 DataFrame: [time, ao_cpc]
    """
    print_header("LOAD NOAA CPC AO FILE")

    rows = []
    bad_lines = 0
    total_lines = 0

    with open(ao_file, "r") as f:
        for line in f:
            total_lines += 1
            parts = line.split()

            if len(parts) == 4:
                try:
                    year = int(parts[0])
                    month = int(parts[1])
                    day = int(parts[2])
                    val = float(parts[3])

                    if val == -99.000:
                        bad_lines += 1
                        continue

                    rows.append((year, month, day, val))
                except Exception:
                    bad_lines += 1
                    continue
            else:
                bad_lines += 1
                continue

    df_cpc = pd.DataFrame(rows, columns=["year", "month", "day", "ao_cpc"])
    df_cpc["time"] = pd.to_datetime(dict(year=df_cpc.year, month=df_cpc.month, day=df_cpc.day))
    df_cpc = df_cpc[["time", "ao_cpc"]].sort_values("time").reset_index(drop=True)

    print(f"NOAA file: {ao_file}")
    print(f"Total lines read : {total_lines}")
    print(f"Bad/skipped lines: {bad_lines}")
    print(f"Valid rows       : {len(df_cpc)}")
    print(f"Time range       : {df_cpc['time'].min()} -> {df_cpc['time'].max()}")
    print(df_cpc.head())

    return df_cpc


# ============================================================
# 2) OPEN ERA5 DAILY Z1000
#    改成更接近 code B: 用 interp(level=1000)，不是 sel(level=1000)
# ============================================================
def open_era5_daily_z1000():
    print_header("1) FIND ERA5 FILES")
    z_files = find_era5_files(ERA5_Z_DIR, YEAR_MIN, YEAR_MAX)
    print(f"Matched ERA5 files: {len(z_files)}")
    print(f"First file: {z_files[0]}")
    print(f"Last  file: {z_files[-1]}")

    print_header("2) OPEN ERA5 AND EXTRACT 1000 hPa (CODE-B STYLE)")
    ds = xr.open_mfdataset(
        z_files,
        combine="by_coords",
        parallel=True,
        chunks={"time": 180}
    )

    plev = ds["level"]
    plev_last = float(plev.values[-1])
    print(f"ERA5 level last value = {plev_last}")

    if plev_last > 1000:
        target_level = 100000.0
    else:
        target_level = 1000.0

    print(f"Interpolating to target level = {target_level}")

    with ProgressBar():
        Z_raw = (
            ds["z"]
            .interp(level=target_level)
            .sel(latitude=slice(LAT_MAX, LAT_MIN))
            .compute()
        )

    print_da_info("Z_raw before /g check", Z_raw)

    mean_val = float(Z_raw.mean().values)
    if mean_val > 500:
        print(f"🔧 mean(z) = {mean_val:.2f} -> interpreted as geopotential, divide by g")
        Z_1000 = Z_raw / G
    else:
        print(f"🔧 mean(z) = {mean_val:.2f} -> interpreted as geopotential height, no division")
        Z_1000 = Z_raw

    print_header("3) DAILY MEAN")
    Z_daily = Z_1000.resample(time="1D").mean().sortby("time")
    print_da_info("Z_daily", Z_daily)
    return Z_daily


# ============================================================
# 3) CODE A METHOD
# ============================================================
def compute_codeA_ao_from_Zdaily(Z_daily):
    print_header("A) RECOMPUTE AO USING CODE A METHOD")

    Z_base_monthly = Z_daily.sel(time=slice("1979-01-01", "2000-12-31")).resample(time="1MS").mean()
    print_da_info("Z_base_monthly", Z_base_monthly)

    clim_monthly = Z_base_monthly.groupby("time.month").mean("time")
    anom_monthly_base = Z_base_monthly.groupby("time.month") - clim_monthly
    print_da_info("anom_monthly_base", anom_monthly_base)

    coslat = np.cos(np.deg2rad(anom_monthly_base.latitude))
    wgts_2d = np.sqrt(coslat).broadcast_like(anom_monthly_base.isel(time=0))
    print("Code A weights:")
    print(f"  min = {float(wgts_2d.min().values):.6f}")
    print(f"  max = {float(wgts_2d.max().values):.6f}")
    print(f"  dims = {wgts_2d.dims}")
    print(f"  shape = {wgts_2d.shape}")

    solver = XarrayEof(anom_monthly_base, weights=wgts_2d.values)

    pc1_raw = solver.pcs(npcs=1, pcscaling=0).squeeze()
    sigma_monthly = pc1_raw.std(dim="time")
    print(f"sigma_monthly = {float(sigma_monthly.values):.6f}")

    eof1_reg = solver.eofsAsCovariance(neofs=1, pcscaling=1).squeeze()
    print_da_info("eof1_reg (Code A)", eof1_reg)

    polar_mean = float(eof1_reg.sel(latitude=slice(90, 75)).mean().values)
    print(f"Polar mean of eof1_reg (90-75N) = {polar_mean:.6f}")

    if polar_mean > 0:
        flip = -1
        eof1_reg = eof1_reg * -1
        print("🔄 Code A flip = -1")
    else:
        flip = 1
        print("✅ Code A flip = +1")

    Z_base_daily = Z_daily.sel(time=slice("1979-01-01", "2000-12-31"))
    clim_daily = Z_base_daily.groupby("time.dayofyear").mean("time")
    clim_daily_smooth = (
        clim_daily
        .rolling(dayofyear=21, center=True)
        .mean()
        .bfill("dayofyear")
        .ffill("dayofyear")
    )

    anom_daily_all = Z_daily.groupby("time.dayofyear") - clim_daily_smooth
    if "dayofyear" in anom_daily_all.coords:
        anom_daily_all = anom_daily_all.drop_vars("dayofyear")

    print_da_info("anom_daily_all (Code A)", anom_daily_all)

    pseudo_pcs_raw = solver.projectField(anom_daily_all, neofs=1, eofscaling=0).squeeze() * flip
    ao_A = pseudo_pcs_raw / sigma_monthly
    ao_A = clean_time_series_da(ao_A, "ao_A")

    print_da_info("ao_A final", ao_A)
    return ao_A


# ============================================================
# 4) STRICT CODE B METHOD
#    这里尽量逐行贴近你给的 code B
# ============================================================
def compute_codeBstyle_ao_from_Zdaily(Z_daily):
    print_header("B) COMPUTE AO USING STRICT CODE-B METHOD ON THE SAME ERA5 DATA")

    # --------------------------------------------------------
    # code B 本质上用的是 zonal mean 数据（*.zm）
    # --------------------------------------------------------
    if "longitude" in Z_daily.dims:
        var = Z_daily.mean("longitude")
    else:
        var = Z_daily.copy()

    print_da_info("Z_zm before anything", var)

    # --------------------------------------------------------
    # code B: time = nc_fid_in['time']
    # --------------------------------------------------------
    time = var["time"]

    # --------------------------------------------------------
    # code B:
    # if lats[0]>0:
    #     lats_orientation = 'negative'
    # else:
    #     lats_orientation = 'positive'
    #
    # 这里必须在任何 sort / interp 之前判断
    # --------------------------------------------------------
    lats_original = var["latitude"]
    if float(lats_original.isel(latitude=0).values) > 0:
        lats_orientation = "negative"
    else:
        lats_orientation = "positive"

    print(f"Original latitude orientation flag = {lats_orientation}")
    print(f"Original first latitude = {float(lats_original.isel(latitude=0).values)}")
    print(f"Original last  latitude = {float(lats_original.isel(latitude=-1).values)}")

    # --------------------------------------------------------
    # code B:
    # var = var.interp(lat=np.linspace(-90,90,73))
    # lats = lats.interp(lat=np.linspace(-90,90,73))
    #
    # 这里直接按升序目标网格插值
    # 若当前 xarray 版本不接受递减纬度插值，则只为插值临时 sort，
    # 但 lats_orientation 仍保留原始顺序判定
    # --------------------------------------------------------
    lat_target = np.linspace(-90.0, 90.0, 73)

    try:
        var = var.interp(latitude=lat_target)
    except Exception:
        var = var.sortby("latitude").interp(latitude=lat_target)

    # code B 里 lats 也是单独 interp；数值上等价于目标纬度本身
    lats = xr.DataArray(
        lat_target,
        coords={"latitude": lat_target},
        dims=["latitude"],
        name="latitude"
    )

    print_da_info("Z_zm interpolated to 73 lat points", var)

    # --------------------------------------------------------
    # code B:
    # var_anomalies = var.groupby("time.dayofyear") - var.groupby("time.dayofyear").mean("time")
    # --------------------------------------------------------
    var_anomalies = var.groupby("time.dayofyear") - var.groupby("time.dayofyear").mean("time")
    print_da_info("var_anomalies before NH slice", var_anomalies)

    # --------------------------------------------------------
    # code B:
    # var_anomalies = var_anomalies.sel(lat=slice(20,90))
    # lats = lats.sel(lat=slice(20,90))
    # --------------------------------------------------------
    var_anomalies = var_anomalies.sel(latitude=slice(20, 90))
    lats = lats.sel(latitude=slice(20, 90))

    print_da_info("var_anomalies 20-90N", var_anomalies)
    print_da_info("lats 20-90N", lats)

    # --------------------------------------------------------
    # code B:
    # lats = np.array(lats)
    # coslat = np.cos(np.deg2rad(lats).clip(0., 1.))
    # wgts = np.sqrt(coslat)[..., np.newaxis]
    #
    # 注意：
    # 你贴出的 code B 里 wgts 最后多了一个 singleton 维。
    # 对于 (time, lat) 的标准 EOF 输入，这种 (lat,1) 权重通常不能直接广播。
    # 所以下面保留“完全相同的权重公式”，但传给 solver 的是数值等价的 1D 权重。
    # --------------------------------------------------------
    lats_np = np.array(lats)
    coslat = np.cos(np.deg2rad(lats_np).clip(0.0, 1.0))

    wgts_codeB_formula = np.sqrt(coslat)          # 等价核心权重
    wgts_codeB_visual  = np.sqrt(coslat)[..., np.newaxis]  # 仅用于展示与 code B 文字一致

    print("Code-B literal weight formula check:")
    print(f"  lats_np shape            = {lats_np.shape}")
    print(f"  coslat min/max          = {np.nanmin(coslat):.6f} / {np.nanmax(coslat):.6f}")
    print(f"  wgts_codeB_formula shape= {wgts_codeB_formula.shape}")
    print(f"  wgts_codeB_visual  shape= {wgts_codeB_visual.shape}")

    # --------------------------------------------------------
    # code B:
    # gh_layer = np.array(var_anomalies)
    # if len(np.shape(gh_layer)) == 3:
    #     gh_layer = np.reshape(gh_layer, (np.shape(gh_layer)[0], len(lats)))
    # solver = Eof(gh_layer, weights=wgts, center=True)
    # AO_x = np.reshape(solver.pcs(npcs=1, pcscaling=1), (np.shape(gh_layer)[0],))
    #
    # 这里按 code B 逻辑转成 numpy，并用标准 EOF
    # --------------------------------------------------------
    gh_layer = np.array(var_anomalies)

    if len(np.shape(gh_layer)) == 3:
        gh_layer = np.reshape(gh_layer, (np.shape(gh_layer)[0], len(lats_np)))

    print("EOF input diagnostics:")
    print(f"  gh_layer shape = {gh_layer.shape}")
    print(f"  weight shape   = {wgts_codeB_formula.shape}")

    solver = StandardEof(gh_layer, weights=wgts_codeB_formula, center=True)

    AO_x = np.reshape(
        solver.pcs(npcs=1, pcscaling=1),
        (np.shape(gh_layer)[0],)
    )

    ao_B = xr.DataArray(
        AO_x,
        coords=[time],
        dims=["time"],
        name="ao_B"
    )

    ao_B = clean_time_series_da(ao_B, "ao_B")
    print_da_info("ao_B before sign check", ao_B)

    # --------------------------------------------------------
    # code B:
    # nof_lats = len(lats)
    # max_AO = AO_xr.argmax()
    #
    # if lats_orientation=='negative':
    #     if np.nanmean(gh_layer[max_AO,0:int(nof_lats/2)-1]) - np.nanmean(gh_layer[max_AO,int(nof_lats/2)+1:nof_lats-1]) < 0:
    #         AO_xr[:] = -AO_xr[:]
    # else:
    #     if np.nanmean(gh_layer[max_AO,0:int(nof_lats/2)-3]) - np.nanmean(gh_layer[max_AO,int(nof_lats/2)+3:nof_lats-1]) < 0:
    #         AO_xr[:] = -AO_xr[:]
    # --------------------------------------------------------
    nof_lats = len(lats_np)
    max_AO = int(ao_B.argmax().values)

    print(f"max_AO index = {max_AO}")
    print(f"max_AO time  = {pd.to_datetime(ao_B.time.values[max_AO])}")

    if lats_orientation == "negative":
        left_mean = np.nanmean(gh_layer[max_AO, 0:int(nof_lats/2)-1])
        right_mean = np.nanmean(gh_layer[max_AO, int(nof_lats/2)+1:nof_lats-1])

        print("Using code-B sign branch: lats_orientation == 'negative'")
        print(f"left_mean  = {left_mean:.6f}")
        print(f"right_mean = {right_mean:.6f}")
        print(f"left-right = {left_mean - right_mean:.6f}")

        if (left_mean - right_mean) < 0:
            ao_B[:] = -ao_B[:]
            print("🔄 AO_B flipped (negative branch)")
        else:
            print("✅ AO_B kept sign (negative branch)")

    else:
        left_mean = np.nanmean(gh_layer[max_AO, 0:int(nof_lats/2)-3])
        right_mean = np.nanmean(gh_layer[max_AO, int(nof_lats/2)+3:nof_lats-1])

        print("Using code-B sign branch: lats_orientation != 'negative'")
        print(f"left_mean  = {left_mean:.6f}")
        print(f"right_mean = {right_mean:.6f}")
        print(f"left-right = {left_mean - right_mean:.6f}")

        if (left_mean - right_mean) < 0:
            ao_B[:] = -ao_B[:]
            print("🔄 AO_B flipped (positive branch)")
        else:
            print("✅ AO_B kept sign (positive branch)")

    print_da_info("ao_B final", ao_B)
    return ao_B


# ============================================================
# 5) MAIN
# ============================================================
Z_daily = open_era5_daily_z1000()

print_header("4) LOAD AO_A IF AVAILABLE, OTHERWISE RECOMPUTE")

recompute_A = True
df_A = None

if AO_A_PATH is not None:
    try:
        df_A = load_saved_ao(AO_A_PATH)
        df_A = maybe_filter_time_df(df_A, COMPARE_START, COMPARE_END)
        recompute_A = False
        print(f"✅ 成功读取已保存 AO_A: {AO_A_PATH}")
    except Exception as e:
        print(f"⚠️ 读取 AO_A 失败，将改为重算。原因: {e}")

if recompute_A:
    ao_A_da = compute_codeA_ao_from_Zdaily(Z_daily)
    ao_A_da = maybe_filter_time_da(ao_A_da, COMPARE_START, COMPARE_END)
    ao_A_da.to_netcdf(OUT_A_AO_NC)
    print(f"✅ 重算后的 AO_A 已保存到: {OUT_A_AO_NC}")

    df_A = pd.DataFrame({
        "time": pd.to_datetime(ao_A_da.time.values),
        "ao_A": ao_A_da.values
    }).dropna().sort_values("time").reset_index(drop=True)

print(df_A.head())
print(f"AO_A length = {len(df_A)}")
print(f"AO_A time range = {df_A['time'].min()} to {df_A['time'].max()}")

ao_B_da = compute_codeBstyle_ao_from_Zdaily(Z_daily)
ao_B_da = maybe_filter_time_da(ao_B_da, COMPARE_START, COMPARE_END)
ao_B_da.to_netcdf(OUT_B_AO_NC)
print(f"✅ AO_B 已保存到: {OUT_B_AO_NC}")

df_B = pd.DataFrame({
    "time": pd.to_datetime(ao_B_da.time.values),
    "ao_B": ao_B_da.values
}).dropna().sort_values("time").reset_index(drop=True)

print(df_B.head())
print(f"AO_B length = {len(df_B)}")
print(f"AO_B time range = {df_B['time'].min()} to {df_B['time'].max()}")

# ============================================================
# 6) MERGE FOR FULL-PERIOD SCATTER
# ============================================================
print_header("5) MERGE ON IDENTICAL DATES FOR A vs B")

df_merge = pd.merge(df_A, df_B, on="time", how="inner").dropna().sort_values("time").reset_index(drop=True)

if len(df_merge) == 0:
    raise ValueError("AO_A 与 AO_B 没有共同日期，无法比较。")

print(df_merge.head())
print(f"Merged length = {len(df_merge)}")
print(f"Merged time range = {df_merge['time'].min()} to {df_merge['time'].max()}")

df_merge.to_csv(OUT_MERGEDCSV, index=False)
print(f"✅ merged CSV 已保存到: {OUT_MERGEDCSV}")

# ============================================================
# 7) CORRELATION + FULL-PERIOD SCATTER
# ============================================================
print_header("6) CORRELATION")

r_val, p_val = pearsonr(df_merge["ao_A"], df_merge["ao_B"])
print(f"Pearson r = {r_val:.6f}")
print(f"p-value   = {p_val:.6e}")

slope, intercept = np.polyfit(df_merge["ao_A"], df_merge["ao_B"], 1)
print(f"Linear fit: ao_B = {slope:.4f} * ao_A + {intercept:.4f}")

print_header("7) PLOT FULL-PERIOD SCATTER")

fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(
    df_merge["ao_A"],
    df_merge["ao_B"],
    s=18,
    alpha=0.55,
    edgecolors="k",
    linewidths=0.3
)

xy_min = min(df_merge["ao_A"].min(), df_merge["ao_B"].min())
xy_max = max(df_merge["ao_A"].max(), df_merge["ao_B"].max())
pad = 0.08 * (xy_max - xy_min)
xy_min -= pad
xy_max += pad

ax.plot([xy_min, xy_max], [xy_min, xy_max], "k--", lw=1.5, label="1:1 line")

xfit = np.linspace(xy_min, xy_max, 200)
yfit = slope * xfit + intercept
ax.plot(xfit, yfit, color="red", lw=2, label=f"Fit: y={slope:.2f}x+{intercept:.2f}")

ax.set_xlim(xy_min, xy_max)
ax.set_ylim(xy_min, xy_max)
ax.set_aspect("equal", adjustable="box")

ax.set_xlabel("AO from Code A", fontsize=13)
ax.set_ylabel("AO from strict Code-B method", fontsize=13)
ax.set_title(
    f"AO Comparison on Identical Dates\n"
    f"Pearson r = {r_val:.3f}, p = {p_val:.2e}, N = {len(df_merge)}",
    fontsize=14,
    fontweight="bold"
)
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
fig.savefig(OUT_SCATTER, dpi=300, bbox_inches="tight")
print(f"✅ 散点图已保存到: {OUT_SCATTER}")

# ============================================================
# 8) NOAA + SHORT-WINDOW TIMESERIES (2019-11 to 2020-06)
# ============================================================
df_cpc = load_noaa_ao_txt(AO_FILE)

print_header("8) PREPARE SHORT-WINDOW TIMESERIES DATA")

df_plot_ab = df_merge[(df_merge["time"] >= pd.to_datetime(PLOT_START)) &
                      (df_merge["time"] <= pd.to_datetime(PLOT_END))].copy()

df_plot_cpc = df_cpc[(df_cpc["time"] >= pd.to_datetime(PLOT_START)) &
                     (df_cpc["time"] <= pd.to_datetime(PLOT_END))].copy()

df_plot = pd.merge(df_plot_ab, df_plot_cpc, on="time", how="inner").dropna().sort_values("time").reset_index(drop=True)

print(f"PLOT_START = {PLOT_START}")
print(f"PLOT_END   = {PLOT_END}")
print(f"AB plot rows   = {len(df_plot_ab)}")
print(f"CPC plot rows  = {len(df_plot_cpc)}")
print(f"Final plot rows= {len(df_plot)}")

if len(df_plot) == 0:
    raise ValueError(f"在 {PLOT_START} 到 {PLOT_END} 之间没有匹配到 A/B/CPC 三者共同时间。")

print(df_plot.head())
print(df_plot.tail())

r_A_CPC, p_A_CPC = pearsonr(df_plot["ao_A"], df_plot["ao_cpc"])
r_B_CPC, p_B_CPC = pearsonr(df_plot["ao_B"], df_plot["ao_cpc"])
r_A_B_window, p_A_B_window = pearsonr(df_plot["ao_A"], df_plot["ao_B"])

print("Window correlations:")
print(f"  A vs CPC: r = {r_A_CPC:.4f}, p = {p_A_CPC:.3e}")
print(f"  B vs CPC: r = {r_B_CPC:.4f}, p = {p_B_CPC:.3e}")
print(f"  A vs B  : r = {r_A_B_window:.4f}, p = {p_A_B_window:.3e}")

print_header("9) PLOT SHORT-WINDOW TIMESERIES")

fig2, ax2 = plt.subplots(figsize=(12, 8))

ax2.plot(df_plot["time"], df_plot["ao_cpc"], lw=2.6, color="black", label="NOAA CPC Official")
ax2.plot(df_plot["time"], df_plot["ao_A"], lw=1.8, color="red", linestyle="--", label="Code A")
ax2.plot(df_plot["time"], df_plot["ao_B"], lw=1.8, color="blue", linestyle="-.", label="Strict Code B")

ax2.axhline(0, color="gray", lw=1.0)
ax2.set_ylabel("Standardized AO Index", fontsize=14)
ax2.set_xlabel("Time", fontsize=14)
ax2.set_title(
    f"AO Comparison ({PLOT_START} to {PLOT_END})\n"
    f"A vs CPC r={r_A_CPC:.3f}, B vs CPC r={r_B_CPC:.3f}, A vs B r={r_A_B_window:.3f}",
    fontsize=15,
    fontweight="bold"
)
ax2.legend(loc="upper right", fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig2.savefig(OUT_TS, dpi=300, bbox_inches="tight")
print(f"✅ 短时间窗折线图已保存到: {OUT_TS}")

print("\n全部完成。")

In [ ]:
# ============================================================
# Standalone script:
# Plot only spatial patterns of AO from
#   A) Code-A method (2D EOF regression/covariance map)
#   B) Strict Code-B method (zonal-mean EOF pattern -> expanded to 2D ring map)
#
# Outputs:
#   1) AO_CodeA_regmap.nc
#   2) AO_CodeB_latpattern.nc
#   3) AO_CodeB_ring_regmap.nc
#   4) AO_spatial_pattern_compare_A_vs_Bring.png
#   5) AO_spatial_pattern_latprofile_A_vs_B.png
#   6) EOF1 explained variance text files for A and B
# ============================================================

import os
import re
import glob
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import dask

from dask.diagnostics import ProgressBar
from eofs.xarray import Eof as XarrayEof
from eofs.standard import Eof as StandardEof

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

dask.config.set(scheduler="threads", num_workers=8)

# ============================================================
# 0) PATHS & CONFIG
# ============================================================
ERA5_Z_DIR = "/pool/datos/reanalisis/era5/6hourly/global"

OUT_DIR = "/home/weiji/restart_exam/code/20260331_AO_spatial_patterns_only"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_A_MAP_NC      = os.path.join(OUT_DIR, "AO_CodeA_regmap.nc")
OUT_B_LAT_NC      = os.path.join(OUT_DIR, "AO_CodeB_latpattern.nc")
OUT_B_RING_NC     = os.path.join(OUT_DIR, "AO_CodeB_ring_regmap.nc")
OUT_MAP_COMPARE   = os.path.join(OUT_DIR, "AO_spatial_pattern_compare_A_vs_Bring.png")
OUT_LATPROFILE    = os.path.join(OUT_DIR, "AO_spatial_pattern_latprofile_A_vs_B.png")

OUT_A_VARFRAC_TXT = os.path.join(OUT_DIR, "AO_CodeA_EOF1_explained_variance.txt")
OUT_B_VARFRAC_TXT = os.path.join(OUT_DIR, "AO_CodeB_EOF1_explained_variance.txt")

G = 9.80665
LAT_MIN = 20.0
LAT_MAX = 90.0
YEAR_MIN = 1950
YEAR_MAX = 2020

BASE_START = "1979-01-01"
BASE_END   = "2000-12-31"

# ============================================================
# 1) HELPERS
# ============================================================
def print_header(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def print_da_info(name, da):
    print(f"[{name}]")
    print(f"  type   : {type(da)}")
    print(f"  dims   : {da.dims}")
    print(f"  shape  : {da.shape}")
    if "time" in da.coords:
        try:
            print(f"  time   : {str(da.time.values[0])} -> {str(da.time.values[-1])}")
        except Exception:
            pass
    if "latitude" in da.coords:
        latv = da.latitude.values
        print(f"  lat    : {latv[0]} -> {latv[-1]}  (n={len(latv)})")
    if "longitude" in da.coords:
        lonv = da.longitude.values
        print(f"  lon    : {lonv[0]} -> {lonv[-1]}  (n={len(lonv)})")
    try:
        arr = da.values
        print(f"  min/max: {np.nanmin(arr):.6f} / {np.nanmax(arr):.6f}")
    except Exception:
        pass
    print()

def save_varfrac_txt(out_txt, method_name, varfrac):
    with open(out_txt, "w") as f:
        f.write(f"{method_name} EOF1 explained variance fraction = {varfrac:.8f}\n")
        f.write(f"{method_name} EOF1 explained variance percent  = {varfrac * 100:.4f}%\n")
    print(f"✅ {method_name} EOF1 explained variance saved to: {out_txt}")

def find_era5_files(era5_dir, year_min=1950, year_max=2020):
    pattern = re.compile(r"^z_4daily_(\d{8})-(\d{8})\.nc$")
    all_files = sorted(glob.glob(os.path.join(era5_dir, "z_4daily*.nc")))

    z_files = []
    for f in all_files:
        base = os.path.basename(f)
        m = pattern.match(base)
        if m is None:
            continue
        y0 = int(m.group(1)[:4])
        if year_min <= y0 <= year_max:
            z_files.append(f)

    if len(z_files) == 0:
        raise FileNotFoundError("没有匹配到 ERA5 z_4daily 文件，请检查路径。")

    return z_files

def get_nh_slice(lat_coord, lat_min=20.0, lat_max=90.0):
    first_lat = float(lat_coord.values[0])
    last_lat  = float(lat_coord.values[-1])

    if first_lat > last_lat:
        # descending latitude, e.g. 90 -> -90
        return slice(lat_max, lat_min)
    else:
        # ascending latitude, e.g. -90 -> 90
        return slice(lat_min, lat_max)

def detect_target_level(level_coord):
    last_val = float(level_coord.values[-1])
    if last_val > 1000:
        return 100000.0   # Pa
    else:
        return 1000.0     # hPa

def extract_z1000(da_z, target_level):
    level_vals = da_z["level"].values.astype(float)

    if np.any(np.isclose(level_vals, target_level)):
        print(f"Using exact level selection: {target_level}")
        z_lev = da_z.sel(level=target_level)
    else:
        print(f"Exact level {target_level} not found; using interpolation")
        z_lev = da_z.interp(level=target_level)

    return z_lev

def maybe_convert_geopotential_to_height(z_da):
    mean_val = float(z_da.mean().values)

    if mean_val > 500:
        print(f"🔧 mean(z) = {mean_val:.2f} -> interpreted as geopotential, divide by g")
        return z_da / G
    else:
        print(f"🔧 mean(z) = {mean_val:.2f} -> interpreted as geopotential height, no division")
        return z_da

def expand_lat_pattern_to_2d(lat_pattern, lon_coord, name="pattern_2d"):
    arr2d = np.repeat(lat_pattern.values[:, np.newaxis], len(lon_coord), axis=1)

    da2d = xr.DataArray(
        arr2d,
        coords={
            "latitude": lat_pattern["latitude"].values,
            "longitude": lon_coord.values
        },
        dims=("latitude", "longitude"),
        name=name
    )
    return da2d

# ============================================================
# 2) OPEN DATA FOR CODE A (NH 20-90, FULL LON, DAILY)
# ============================================================
def open_era5_daily_z1000_nh2d():
    print_header("1) OPEN ERA5 DAILY Z1000 FOR CODE A (NH 20-90, FULL LON)")

    z_files = find_era5_files(ERA5_Z_DIR, YEAR_MIN, YEAR_MAX)
    print(f"Matched ERA5 files: {len(z_files)}")
    print(f"First file: {z_files[0]}")
    print(f"Last  file: {z_files[-1]}")

    ds = xr.open_mfdataset(
        z_files,
        combine="by_coords",
        parallel=True,
        chunks={"time": 180}
    )

    target_level = detect_target_level(ds["level"])
    print(f"Detected target level = {target_level}")

    z_1000 = extract_z1000(ds["z"], target_level)

    nh_slice = get_nh_slice(z_1000["latitude"], LAT_MIN, LAT_MAX)

    with ProgressBar():
        z_nh = z_1000.sel(latitude=nh_slice).compute()

    z_nh = maybe_convert_geopotential_to_height(z_nh)

    print_header("DAILY MEAN FOR CODE A")
    z_daily = z_nh.resample(time="1D").mean().sortby("time")
    print_da_info("Z_daily_A", z_daily)

    return z_daily

# ============================================================
# 3) OPEN DATA FOR CODE B (GLOBAL ZONAL MEAN, DAILY)
# ============================================================
def open_era5_daily_z1000_global_zm():
    print_header("2) OPEN ERA5 DAILY Z1000 FOR CODE B (GLOBAL ZONAL MEAN)")

    z_files = find_era5_files(ERA5_Z_DIR, YEAR_MIN, YEAR_MAX)

    ds = xr.open_mfdataset(
        z_files,
        combine="by_coords",
        parallel=True,
        chunks={"time": 180}
    )

    target_level = detect_target_level(ds["level"])
    print(f"Detected target level = {target_level}")

    z_1000 = extract_z1000(ds["z"], target_level)

    with ProgressBar():
        z_zm = z_1000.mean("longitude").compute()

    z_zm = maybe_convert_geopotential_to_height(z_zm)

    print_header("DAILY MEAN FOR CODE B")
    z_daily_zm = z_zm.resample(time="1D").mean().sortby("time")
    print_da_info("Z_daily_B_zm", z_daily_zm)

    return z_daily_zm

# ============================================================
# 4) CODE A SPATIAL PATTERN
#    2D EOF regression/covariance map (m)
# ============================================================
def compute_codeA_spatial_pattern(Z_daily_A):
    print_header("3) COMPUTE CODE-A SPATIAL PATTERN")

    Z_base_monthly = Z_daily_A.sel(time=slice(BASE_START, BASE_END)).resample(time="1MS").mean()
    print_da_info("Z_base_monthly_A", Z_base_monthly)

    clim_monthly = Z_base_monthly.groupby("time.month").mean("time")
    anom_monthly_base = Z_base_monthly.groupby("time.month") - clim_monthly
    print_da_info("anom_monthly_base_A", anom_monthly_base)

    coslat = np.cos(np.deg2rad(anom_monthly_base.latitude))
    wgts_2d = np.sqrt(coslat).broadcast_like(anom_monthly_base.isel(time=0))

    solver = XarrayEof(anom_monthly_base, weights=wgts_2d.values)

    # EOF1 explained variance fraction
    varfrac_A = float(solver.varianceFraction(neigs=1).squeeze().values)
    print(f"Code A EOF1 explained variance fraction = {varfrac_A:.8f}")
    print(f"Code A EOF1 explained variance percent  = {varfrac_A * 100:.4f}%")

    pc1_raw = solver.pcs(npcs=1, pcscaling=0).squeeze()
    sigma_monthly = pc1_raw.std(dim="time")
    print(f"sigma_monthly (Code A) = {float(sigma_monthly.values):.6f}")

    eof1_reg_A = solver.eofsAsCovariance(neofs=1, pcscaling=1).squeeze()
    eof1_reg_A.name = "eof1_reg_A"
    print_da_info("eof1_reg_A before sign", eof1_reg_A)

    polar_mean = float(eof1_reg_A.sel(latitude=get_nh_slice(eof1_reg_A.latitude, 75, 90)).mean().values)
    print(f"Polar mean of eof1_reg_A over 75-90N = {polar_mean:.6f}")

    if polar_mean > 0:
        eof1_reg_A = eof1_reg_A * -1
        print("🔄 Code A pattern flipped")
    else:
        print("✅ Code A pattern kept sign")

    print_da_info("eof1_reg_A final", eof1_reg_A)
    return eof1_reg_A, varfrac_A

# ============================================================
# 5) STRICT CODE-B SPATIAL PATTERN
#    1D zonal-mean EOF regression/covariance pattern (m)
#    then expand to 2D ring pattern
# ============================================================
def compute_codeB_ring_pattern(Z_daily_B_zm, lon_template):
    print_header("4) COMPUTE STRICT CODE-B SPATIAL PATTERN")

    var = Z_daily_B_zm.copy()
    print_da_info("Z_daily_B_zm input", var)

    time = var["time"]

    # Must determine latitude orientation BEFORE sort/interp
    lats_original = var["latitude"]
    if float(lats_original.isel(latitude=0).values) > 0:
        lats_orientation = "negative"
    else:
        lats_orientation = "positive"

    print(f"Original latitude orientation flag = {lats_orientation}")
    print(f"Original first latitude = {float(lats_original.isel(latitude=0).values)}")
    print(f"Original last  latitude = {float(lats_original.isel(latitude=-1).values)}")

    # Strict Code-B-like interpolation to 73 lat points from -90 to 90
    lat_target = np.linspace(-90.0, 90.0, 73)

    try:
        var_interp = var.interp(latitude=lat_target)
    except Exception:
        var_interp = var.sortby("latitude").interp(latitude=lat_target)

    lats = xr.DataArray(
        lat_target,
        coords={"latitude": lat_target},
        dims=["latitude"],
        name="latitude"
    )

    print_da_info("var_interp_B", var_interp)

    # Code B:
    # var_anomalies = var.groupby("time.dayofyear") - var.groupby("time.dayofyear").mean("time")
    var_anomalies = var_interp.groupby("time.dayofyear") - var_interp.groupby("time.dayofyear").mean("time")
    print_da_info("var_anomalies_B before NH slice", var_anomalies)

    var_anomalies = var_anomalies.sel(latitude=slice(20, 90))
    lats = lats.sel(latitude=slice(20, 90))

    print_da_info("var_anomalies_B 20-90N", var_anomalies)
    print_da_info("lats_B 20-90N", lats)

    # Strictly follow the code-B formula you posted
    lats_np = np.array(lats)
    coslat = np.cos(np.deg2rad(lats_np).clip(0.0, 1.0))
    wgts_codeB_formula = np.sqrt(coslat)

    print("Code-B weight diagnostics:")
    print(f"  lats_np shape = {lats_np.shape}")
    print(f"  coslat min/max = {np.nanmin(coslat):.6f} / {np.nanmax(coslat):.6f}")
    print(f"  wgts shape = {wgts_codeB_formula.shape}")

    gh_layer = np.array(var_anomalies)

    if len(np.shape(gh_layer)) == 3:
        gh_layer = np.reshape(gh_layer, (np.shape(gh_layer)[0], len(lats_np)))

    print("EOF input diagnostics (Code B):")
    print(f"  gh_layer shape = {gh_layer.shape}")
    print(f"  weight shape   = {wgts_codeB_formula.shape}")

    solver = StandardEof(gh_layer, weights=wgts_codeB_formula, center=True)

    # EOF1 explained variance fraction
    varfrac_B = float(np.squeeze(solver.varianceFraction(neigs=1)))
    print(f"Code B EOF1 explained variance fraction = {varfrac_B:.8f}")
    print(f"Code B EOF1 explained variance percent  = {varfrac_B * 100:.4f}%")

    AO_x = np.reshape(
        solver.pcs(npcs=1, pcscaling=1),
        (np.shape(gh_layer)[0],)
    )

    ao_B = xr.DataArray(
        AO_x,
        coords=[time],
        dims=["time"],
        name="ao_B_for_sign"
    )

    eof1_reg_B_lat = solver.eofsAsCovariance(neofs=1, pcscaling=1).squeeze()
    eof1_reg_B_lat = xr.DataArray(
        eof1_reg_B_lat,
        coords={"latitude": lats.values},
        dims=["latitude"],
        name="eof1_reg_B_lat"
    )

    print_da_info("eof1_reg_B_lat before sign", eof1_reg_B_lat)

    # Strict Code-B sign logic
    nof_lats = len(lats_np)
    max_AO = int(ao_B.argmax().values)

    print(f"max_AO index = {max_AO}")
    print(f"max_AO time  = {str(ao_B.time.values[max_AO])}")

    if lats_orientation == "negative":
        left_mean = np.nanmean(gh_layer[max_AO, 0:int(nof_lats/2)-1])
        right_mean = np.nanmean(gh_layer[max_AO, int(nof_lats/2)+1:nof_lats-1])

        print("Using Code-B sign branch: lats_orientation == 'negative'")
        print(f"left_mean  = {left_mean:.6f}")
        print(f"right_mean = {right_mean:.6f}")
        print(f"left-right = {left_mean - right_mean:.6f}")

        if (left_mean - right_mean) < 0:
            flip_B = -1
            print("🔄 Code B pattern flipped (negative branch)")
        else:
            flip_B = 1
            print("✅ Code B pattern kept sign (negative branch)")
    else:
        left_mean = np.nanmean(gh_layer[max_AO, 0:int(nof_lats/2)-3])
        right_mean = np.nanmean(gh_layer[max_AO, int(nof_lats/2)+3:nof_lats-1])

        print("Using Code-B sign branch: lats_orientation != 'negative'")
        print(f"left_mean  = {left_mean:.6f}")
        print(f"right_mean = {right_mean:.6f}")
        print(f"left-right = {left_mean - right_mean:.6f}")

        if (left_mean - right_mean) < 0:
            flip_B = -1
            print("🔄 Code B pattern flipped (positive branch)")
        else:
            flip_B = 1
            print("✅ Code B pattern kept sign (positive branch)")

    eof1_reg_B_lat = eof1_reg_B_lat * flip_B
    print_da_info("eof1_reg_B_lat final", eof1_reg_B_lat)

    eof1_reg_B_ring = expand_lat_pattern_to_2d(
        eof1_reg_B_lat,
        lon_template,
        name="eof1_reg_B_ring"
    )
    print_da_info("eof1_reg_B_ring", eof1_reg_B_ring)

    return eof1_reg_B_lat, eof1_reg_B_ring, varfrac_B

# ============================================================
# 6) PLOT: TWO POLAR MAPS
# ============================================================
def plot_two_polar_maps(mapA, mapB, varfrac_A, varfrac_B, out_png):
    print_header("5) PLOT TWO POLAR SPATIAL PATTERNS")

    # Sort latitude ascending for plotting stability
    mapA_plot = mapA.sortby("latitude")
    mapB_plot = mapB.sortby("latitude")

    # NOAA-like discrete palette
    noaa_colors = [
        "#2d1e8f",  # < -40
        "#4355c7",  # -40 to -35
        "#5a8ce6",  # -35 to -30
        "#7ebbf5",  # -30 to -25
        "#a3dcfb",  # -25 to -20
        "#c2f0fb",  # -20 to -15
        "#dff8fd",  # -15 to -10
        "#f0fcfd",  # -10 to -5
        "#ffffff",  # -5 to 5
        "#fefbd9",  # 5 to 10
        "#feea9e",  # 10 to 15
        "#fec762",  # 15 to 20
        "#f27932",  # 20 to 25
        "#b82522"   # > 25
    ]
    bounds = [-45, -40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25, 30]
    cbar_ticks = [-40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25]

    cmap_noaa = mcolors.ListedColormap(noaa_colors)
    norm_noaa = mcolors.BoundaryNorm(bounds, cmap_noaa.N)

    fig, axes = plt.subplots(
        1, 2, figsize=(14, 7),
        subplot_kw={"projection": ccrs.NorthPolarStereo()}
    )

    panel_titles = [
        f"Code A: 2D EOF regression/covariance pattern\nEOF1 explained variance = {varfrac_A * 100:.2f}%",
        f"Strict Code B: zonal-mean EOF expanded to 2D ring pattern\nEOF1 explained variance = {varfrac_B * 100:.2f}%"
    ]

    for ax, da, ttl in zip(axes, [mapA_plot, mapB_plot], panel_titles):
        ax.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())
        ax.coastlines(linewidth=1)
        ax.add_feature(cfeature.BORDERS, linestyle=":", alpha=0.5)

        lon = da.longitude.values
        lat = da.latitude.values
        field_cyc, lon_cyc = add_cyclic_point(da.values, coord=lon)

        cf = ax.contourf(
            lon_cyc, lat, field_cyc,
            levels=bounds,
            transform=ccrs.PlateCarree(),
            cmap=cmap_noaa,
            norm=norm_noaa,
            extend="both"
        )
        ax.set_title(ttl, fontsize=12, fontweight="bold", pad=14)

    fig.suptitle(
        "AO Spatial Patterns: Code A vs Strict Code-B Ring Pattern",
        fontsize=15,
        fontweight="bold",
        y=0.95
    )

    # Manually place the two map panels higher, leaving room for colorbar
    fig.subplots_adjust(left=0.05, right=0.95, top=0.86, bottom=0.18, wspace=0.14)

    # Independent colorbar axis below the plots
    cbar_ax = fig.add_axes([0.20, 0.08, 0.60, 0.035])
    cbar = fig.colorbar(
        cf,
        cax=cbar_ax,
        orientation="horizontal",
        ticks=cbar_ticks
    )
    cbar.set_label("Regression / covariance map of 1000 hPa height (m)", fontsize=12)

    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    print(f"✅ 空间构型对比图已保存到: {out_png}")

# ============================================================
# 7) PLOT: LATITUDE PROFILE COMPARISON
# ============================================================
def plot_lat_profiles(mapA, lat_pattern_B, out_png):
    print_header("6) PLOT LATITUDE PROFILE COMPARISON")

    A_zm = mapA.mean("longitude").sortby("latitude")
    B_lat = lat_pattern_B.sortby("latitude")

    # Interpolate B onto A grid for easy comparison
    B_on_A = B_lat.interp(latitude=A_zm.latitude)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(A_zm.latitude, A_zm, lw=2.2, color="red", label="Code A zonal-mean profile")
    ax.plot(B_on_A.latitude, B_on_A, lw=2.2, color="blue", linestyle="--", label="Code B 1D lat pattern")

    ax.axhline(0, color="gray", lw=1.0)
    ax.set_xlim(20, 90)
    ax.set_xlabel("Latitude (°N)", fontsize=10)
    ax.set_ylabel("Regression / covariance height anomaly (m)", fontsize=10)
    ax.set_title("AO spatial pattern latitude profile comparison", fontsize=12, fontweight="bold")
    ax.grid(True, alpha=0.3)
    ax.legend()

    plt.tight_layout()
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    print(f"✅ 纬向剖面对比图已保存到: {out_png}")

# ============================================================
# 8) MAIN
# ============================================================
def main():
    # Code A input: NH 20-90, full longitude
    Z_daily_A = open_era5_daily_z1000_nh2d()

    # Code B input: global zonal mean only
    Z_daily_B_zm = open_era5_daily_z1000_global_zm()

    # Spatial patterns
    eof1_reg_A, varfrac_A = compute_codeA_spatial_pattern(Z_daily_A)
    eof1_reg_B_lat, eof1_reg_B_ring, varfrac_B = compute_codeB_ring_pattern(
        Z_daily_B_zm,
        lon_template=Z_daily_A.longitude
    )

    # Save netCDF
    eof1_reg_A.to_netcdf(OUT_A_MAP_NC)
    print(f"✅ Code A 空间构型已保存到: {OUT_A_MAP_NC}")

    eof1_reg_B_lat.to_netcdf(OUT_B_LAT_NC)
    print(f"✅ Code B 1D 纬向构型已保存到: {OUT_B_LAT_NC}")

    eof1_reg_B_ring.to_netcdf(OUT_B_RING_NC)
    print(f"✅ Code B 2D 环状构型已保存到: {OUT_B_RING_NC}")

    # Save explained variance
    save_varfrac_txt(OUT_A_VARFRAC_TXT, "Code A", varfrac_A)
    save_varfrac_txt(OUT_B_VARFRAC_TXT, "Code B", varfrac_B)

    # Plot
    plot_two_polar_maps(eof1_reg_A, eof1_reg_B_ring, varfrac_A, varfrac_B, OUT_MAP_COMPARE)
    plot_lat_profiles(eof1_reg_A, eof1_reg_B_lat, OUT_LATPROFILE)

    print("\n全部完成。")

if __name__ == "__main__":
    main()

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

OUT_DIR = "/home/weiji/restart_exam/code/20260331_AO_spatial_patterns_only"

A_MAP = f"{OUT_DIR}/AO_CodeA_regmap.nc"
B_RING = f"{OUT_DIR}/AO_CodeB_ring_regmap.nc"
OUT_PNG = f"{OUT_DIR}/AO_spatial_pattern_compare_A_vs_Bring_replot.png"

def plot_two_polar_maps(mapA, mapB, out_png):
    mapA_plot = mapA.sortby("latitude")
    mapB_plot = mapB.sortby("latitude")

    noaa_colors = [
        "#2d1e8f", "#4355c7", "#5a8ce6", "#7ebbf5",
        "#a3dcfb", "#c2f0fb", "#dff8fd", "#f0fcfd",
        "#ffffff", "#fefbd9", "#feea9e", "#fec762",
        "#f27932", "#b82522"
    ]
    bounds = [-45, -40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25, 30]

    cmap_noaa = mcolors.ListedColormap(noaa_colors)
    norm_noaa = mcolors.BoundaryNorm(bounds, cmap_noaa.N)

    fig, axes = plt.subplots(
        1, 2, figsize=(14, 7),
        subplot_kw={"projection": ccrs.NorthPolarStereo()}
    )

    for ax, da, ttl in zip(
        axes,
        [mapA_plot, mapB_plot],
        [
            "Code A: 2D EOF regression/covariance pattern",
            "Strict Code B: zonal-mean EOF expanded to 2D ring pattern"
        ]
    ):
        ax.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())
        ax.coastlines(linewidth=1)
        ax.add_feature(cfeature.BORDERS, linestyle=":", alpha=0.5)

        lon = da.longitude.values
        lat = da.latitude.values
        field_cyc, lon_cyc = add_cyclic_point(da.values, coord=lon)

        cf = ax.contourf(
            lon_cyc, lat, field_cyc,
            levels=bounds,
            transform=ccrs.PlateCarree(),
            cmap=cmap_noaa,
            norm=norm_noaa,
            extend="both"
        )
        ax.set_title(ttl, fontsize=12, fontweight="bold", pad=14)

    cbar_ticks = [-40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25]
    cbar_ax = fig.add_axes([0.2, 0.03, 0.6, 0.025])
    cbar = fig.colorbar(cf, cax=cbar_ax, orientation="horizontal", ticks=cbar_ticks)
    cbar.set_label("Regression / covariance map of 1000 hPa height (m)", fontsize=12)

    fig.suptitle(
        "AO Spatial Patterns: Code A vs Strict Code-B Ring Pattern",
        fontsize=15, fontweight="bold", y=0.98
    )

    fig.subplots_adjust(left=0.05, right=0.95, top=0.88, bottom=0.14, wspace=0.12)

    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()

dsA = xr.open_dataset(A_MAP)
dsB = xr.open_dataset(B_RING)

mapA = dsA[list(dsA.data_vars)[0]]
mapB = dsB[list(dsB.data_vars)[0]]

plot_two_polar_maps(mapA, mapB, OUT_PNG)
print("saved:", OUT_PNG)

In [ ]:
# ============================================================
# Compare AO from:
#   A) Code-A AO method
#   B) Code-B-style AO method with modified latitude weights
#
# Outputs:
#   1) AO_A (Code A)
#   2) AO_B_modw (Code B with modified weights)
#   3) merged CSV on identical dates
#   4) full-period scatter plot: AO_A vs AO_B_modw
#   5) short-window timeseries plot (2019-11 to 2020-06):
#        AO_A, AO_B_modw, NOAA CPC
#   6) spatial pattern comparison map:
#        Code A 2D pattern vs Code-B ring pattern
#   7) latitude-profile comparison:
#        zonal-mean(Code A pattern) vs Code-B lat pattern
#
# Notes:
#   - The only intended modification to Code B is the latitude weight formula:
#         coslat = np.cos(np.deg2rad(lats)).clip(0.0, 1.0)
#         wgts   = np.sqrt(coslat)
#   - Everything else in the Code-B branch is kept as close as possible
#     to the previously discussed literal Code-B style.
# ============================================================

# ============================================================
# NOTE ON CODE-B WEIGHT MODIFICATION
#
# The original Code-B implementation applies latitude weights as:
#     coslat = np.cos(np.deg2rad(lats).clip(0., 1.))
#     wgts   = np.sqrt(coslat)
#
# This clips the latitude in radians BEFORE applying cosine,
# which effectively flattens all latitudes above ~57°N to nearly
# the same weight. As a result, high-latitude variability is
# artificially over-emphasized, producing an overly symmetric,
# ring-like EOF pattern centered on the pole.
#
# In this script, we correct the weight formulation to:
#     coslat = np.cos(np.deg2rad(lats)).clip(0.0, 1.0)
#     wgts   = np.sqrt(coslat)
#
# i.e. cosine is applied first, and clipping is only used to
# enforce non-negative values. This restores the standard
# latitude-area weighting (sqrt(cos(lat))) and avoids the
# artificial flattening of high-latitude weights.
#
# All other aspects of the Code-B method are kept unchanged,
# so that differences can be attributed solely to the weight
# formulation.
# ============================================================

import os
import re
import glob
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import dask

from dask.diagnostics import ProgressBar
from scipy.stats import pearsonr
from eofs.xarray import Eof as XarrayEof
from eofs.standard import Eof as StandardEof

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

dask.config.set(scheduler="threads", num_workers=4)

# ============================================================
# 0) PATHS & CONFIG
# ============================================================
ERA5_Z_DIR = "/pool/datos/reanalisis/era5/6hourly/global"
AO_FILE = "/mnt/soclim0/public_data/weiji/NOAA_AO_INEX_260228.txt"

OUT_DIR = "/home/weiji/restart_exam/code/20260331_AO_compare_A_vs_Bmodw"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_A_AO_NC = os.path.join(OUT_DIR, "AO_codeA.nc")
OUT_B_AO_NC = os.path.join(OUT_DIR, "AO_codeB_modified_weights.nc")
OUT_MERGEDCSV = os.path.join(OUT_DIR, "AO_compare_A_vs_Bmodw_merged.csv")

OUT_SCATTER = os.path.join(OUT_DIR, "AO_compare_A_vs_Bmodw_scatter.png")
OUT_TS = os.path.join(OUT_DIR, "AO_compare_201911_202006_timeseries_with_NOAA_Bmodw.png")

OUT_A_MAP_NC = os.path.join(OUT_DIR, "AO_codeA_regmap.nc")
OUT_B_LAT_NC = os.path.join(OUT_DIR, "AO_codeB_modified_weights_latpattern.nc")
OUT_B_RING_NC = os.path.join(OUT_DIR, "AO_codeB_modified_weights_ring_regmap.nc")

OUT_MAP_COMPARE = os.path.join(OUT_DIR, "AO_spatial_patterns_A_vs_Bmodw_ring.png")
OUT_LATPROFILE = os.path.join(OUT_DIR, "AO_spatial_pattern_latprofile_A_vs_Bmodw.png")

G = 9.80665
LAT_MIN = 20.0
LAT_MAX = 90.0
YEAR_MIN = 1950
YEAR_MAX = 2020

BASE_START = "1979-01-01"
BASE_END = "2000-12-31"

COMPARE_START = None
COMPARE_END = None

PLOT_START = "2019-11-01"
PLOT_END = "2020-06-30"

# ============================================================
# 1) HELPERS
# ============================================================
def print_header(title):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

def print_da_info(name, da):
    print(f"[{name}]")
    print(f"  type   : {type(da)}")
    print(f"  dims   : {da.dims}")
    print(f"  shape  : {da.shape}")

    if "time" in da.coords:
        try:
            print(f"  time   : {pd.to_datetime(da.time.values[0])} -> {pd.to_datetime(da.time.values[-1])}")
        except Exception:
            pass

    if "latitude" in da.coords:
        latv = da.latitude.values
        print(f"  lat    : {latv[0]} -> {latv[-1]}  (n={len(latv)})")

    if "longitude" in da.coords:
        lonv = da.longitude.values
        print(f"  lon    : {lonv[0]} -> {lonv[-1]}  (n={len(lonv)})")

    try:
        arr = da.values
        print(f"  min/max: {np.nanmin(arr):.6f} / {np.nanmax(arr):.6f}")
        print(f"  NaN cnt: {int(np.isnan(arr).sum())}")
    except Exception:
        pass

    print()

def clean_time_series_da(da, new_name):
    da = da.squeeze(drop=True)

    extra_coord_names = []
    for c in da.coords:
        if c != "time" and c not in da.dims:
            extra_coord_names.append(c)

    if len(extra_coord_names) > 0:
        da = da.drop_vars(extra_coord_names)

    da = da.squeeze(drop=True)

    if da.ndim != 1 or da.dims != ("time",):
        raise ValueError(f"After cleaning, series is not 1D time series: dims={da.dims}, shape={da.shape}")

    da.name = new_name
    return da

def maybe_filter_time_df(df, start=None, end=None):
    out = df.copy()
    if start is not None:
        out = out[out["time"] >= pd.to_datetime(start)]
    if end is not None:
        out = out[out["time"] <= pd.to_datetime(end)]
    return out.reset_index(drop=True)

def maybe_filter_time_da(da, start=None, end=None):
    out = da
    if start is not None and end is not None:
        out = out.sel(time=slice(pd.to_datetime(start), pd.to_datetime(end)))
    elif start is not None:
        out = out.sel(time=slice(pd.to_datetime(start), None))
    elif end is not None:
        out = out.sel(time=slice(None, pd.to_datetime(end)))
    return out

def find_era5_files(era5_dir, year_min=1950, year_max=2020):
    pattern = re.compile(r"^z_4daily_(\d{8})-(\d{8})\.nc$")
    all_files = sorted(glob.glob(os.path.join(era5_dir, "z_4daily*.nc")))

    z_files = []
    for f in all_files:
        base = os.path.basename(f)
        m = pattern.match(base)
        if m is None:
            continue
        y0 = int(m.group(1)[:4])
        if year_min <= y0 <= year_max:
            z_files.append(f)

    if len(z_files) == 0:
        raise FileNotFoundError("No matched ERA5 z_4daily files found.")

    return z_files

def get_lat_slice(lat_coord, lat_min=20.0, lat_max=90.0):
    first_lat = float(lat_coord.values[0])
    last_lat = float(lat_coord.values[-1])

    if first_lat > last_lat:
        return slice(lat_max, lat_min)
    else:
        return slice(lat_min, lat_max)

def detect_target_level(level_coord):
    last_val = float(level_coord.values[-1])
    if last_val > 1000:
        return 100000.0
    else:
        return 1000.0

def extract_z1000_prefer_exact(z_da, target_level):
    level_vals = z_da["level"].values.astype(float)

    if np.any(np.isclose(level_vals, target_level)):
        print(f"Using exact level selection: {target_level}")
        z_lev = z_da.sel(level=target_level)
    else:
        print(f"Exact level {target_level} not found; using interpolation")
        z_lev = z_da.interp(level=target_level)

    return z_lev

def maybe_convert_geopotential_to_height(z_da):
    mean_val = float(z_da.mean().values)
    if mean_val > 500:
        print(f"mean(z) = {mean_val:.2f} -> interpreted as geopotential, divide by g")
        return z_da / G
    else:
        print(f"mean(z) = {mean_val:.2f} -> interpreted as geopotential height, no division")
        return z_da

def load_noaa_ao_txt(ao_file):
    print_header("LOAD NOAA CPC AO FILE")

    rows = []
    total_lines = 0
    bad_lines = 0

    with open(ao_file, "r") as f:
        for line in f:
            total_lines += 1
            parts = line.split()

            if len(parts) == 4:
                try:
                    year = int(parts[0])
                    month = int(parts[1])
                    day = int(parts[2])
                    val = float(parts[3])

                    if val == -99.000:
                        bad_lines += 1
                        continue

                    rows.append((year, month, day, val))
                except Exception:
                    bad_lines += 1
                    continue
            else:
                bad_lines += 1
                continue

    df_cpc = pd.DataFrame(rows, columns=["year", "month", "day", "ao_cpc"])
    df_cpc["time"] = pd.to_datetime(dict(year=df_cpc.year, month=df_cpc.month, day=df_cpc.day))
    df_cpc = df_cpc[["time", "ao_cpc"]].sort_values("time").reset_index(drop=True)

    print(f"NOAA file: {ao_file}")
    print(f"Total lines read : {total_lines}")
    print(f"Bad/skipped lines: {bad_lines}")
    print(f"Valid rows       : {len(df_cpc)}")
    print(f"Time range       : {df_cpc['time'].min()} -> {df_cpc['time'].max()}")
    print(df_cpc.head())

    return df_cpc

def expand_lat_pattern_to_2d(lat_pattern, lon_coord, name="pattern_2d"):
    arr2d = np.repeat(lat_pattern.values[:, np.newaxis], len(lon_coord), axis=1)

    da2d = xr.DataArray(
        arr2d,
        coords={
            "latitude": lat_pattern["latitude"].values,
            "longitude": lon_coord.values
        },
        dims=("latitude", "longitude"),
        name=name
    )
    return da2d

# ============================================================
# 2) OPEN GLOBAL ERA5 DAILY Z1000 ONCE
# ============================================================
def open_era5_daily_z1000_global():
    print_header("1) OPEN GLOBAL ERA5 DAILY Z1000")

    z_files = find_era5_files(ERA5_Z_DIR, YEAR_MIN, YEAR_MAX)
    print(f"Matched ERA5 files: {len(z_files)}")
    print(f"First file: {z_files[0]}")
    print(f"Last  file: {z_files[-1]}")

    ds = xr.open_mfdataset(
        z_files,
        combine="by_coords",
        parallel=True,
        chunks={"time": 180}
    )

    target_level = detect_target_level(ds["level"])
    print(f"Detected target level = {target_level}")

    z_1000_raw = extract_z1000_prefer_exact(ds["z"], target_level)

    with ProgressBar():
        z_1000_raw = z_1000_raw.compute()

    z_1000 = maybe_convert_geopotential_to_height(z_1000_raw)

    print_header("2) DAILY MEAN")
    z_daily = z_1000.resample(time="1D").mean().sortby("time")
    print_da_info("Z_daily_global", z_daily)

    return z_daily

# ============================================================
# 3) CODE A METHOD
# ============================================================
def compute_codeA_ao_and_pattern(Z_daily_global):
    print_header("3) COMPUTE CODE A AO AND SPATIAL PATTERN")

    nh_slice = get_lat_slice(Z_daily_global["latitude"], LAT_MIN, LAT_MAX)
    Z_daily_A = Z_daily_global.sel(latitude=nh_slice)
    print_da_info("Z_daily_A_NH", Z_daily_A)

    Z_base_monthly = Z_daily_A.sel(time=slice(BASE_START, BASE_END)).resample(time="1MS").mean()
    print_da_info("Z_base_monthly_A", Z_base_monthly)

    clim_monthly = Z_base_monthly.groupby("time.month").mean("time")
    anom_monthly_base = Z_base_monthly.groupby("time.month") - clim_monthly
    print_da_info("anom_monthly_base_A", anom_monthly_base)

    coslat = np.cos(np.deg2rad(anom_monthly_base.latitude))
    wgts_2d = np.sqrt(coslat).broadcast_like(anom_monthly_base.isel(time=0))

    solver = XarrayEof(anom_monthly_base, weights=wgts_2d.values)

    # EOF1 explained variance fraction
    varfrac_A = float(solver.varianceFraction(neigs=1).squeeze().values)
    print(f"Code A EOF1 explained variance fraction = {varfrac_A:.6f} ({varfrac_A*100:.2f}%)")

    pc1_raw = solver.pcs(npcs=1, pcscaling=0).squeeze()
    sigma_monthly = pc1_raw.std(dim="time")
    print(f"sigma_monthly (Code A) = {float(sigma_monthly.values):.6f}")

    eof1_reg_A = solver.eofsAsCovariance(neofs=1, pcscaling=1).squeeze()
    eof1_reg_A.name = "eof1_reg_A"
    print_da_info("eof1_reg_A before sign", eof1_reg_A)

    polar_slice_A = get_lat_slice(eof1_reg_A["latitude"], 75.0, 90.0)
    polar_mean = float(eof1_reg_A.sel(latitude=polar_slice_A).mean().values)
    print(f"Polar mean of eof1_reg_A over 75-90N = {polar_mean:.6f}")

    if polar_mean > 0:
        flip_A = -1
        eof1_reg_A = eof1_reg_A * -1
        print("Code A pattern flipped")
    else:
        flip_A = 1
        print("Code A pattern kept sign")

    Z_base_daily = Z_daily_A.sel(time=slice(BASE_START, BASE_END))
    clim_daily = Z_base_daily.groupby("time.dayofyear").mean("time")
    clim_daily_smooth = (
        clim_daily
        .rolling(dayofyear=21, center=True)
        .mean()
        .bfill("dayofyear")
        .ffill("dayofyear")
    )

    anom_daily_all = Z_daily_A.groupby("time.dayofyear") - clim_daily_smooth
    if "dayofyear" in anom_daily_all.coords:
        anom_daily_all = anom_daily_all.drop_vars("dayofyear")

    print_da_info("anom_daily_all_A", anom_daily_all)

    pseudo_pcs_raw = solver.projectField(anom_daily_all, neofs=1, eofscaling=0).squeeze() * flip_A
    ao_A = pseudo_pcs_raw / sigma_monthly
    ao_A = clean_time_series_da(ao_A, "ao_A")

    print_da_info("ao_A final", ao_A)
    print_da_info("eof1_reg_A final", eof1_reg_A)

    return ao_A, eof1_reg_A, Z_daily_A, varfrac_A

# ============================================================
# 4) MODIFIED-WEIGHT CODE B METHOD
# ============================================================
def compute_codeB_modified_weights_ao_and_pattern(Z_daily_global, lon_template):
    print_header("4) COMPUTE MODIFIED-WEIGHT CODE B AO AND SPATIAL PATTERN")

    if "longitude" in Z_daily_global.dims:
        var = Z_daily_global.mean("longitude")
    else:
        var = Z_daily_global.copy()

    print_da_info("Z_daily_B_global_zm", var)

    time = var["time"]

    lats_original = var["latitude"]
    if float(lats_original.isel(latitude=0).values) > 0:
        lats_orientation = "negative"
    else:
        lats_orientation = "positive"

    print(f"Original latitude orientation flag = {lats_orientation}")
    print(f"Original first latitude = {float(lats_original.isel(latitude=0).values)}")
    print(f"Original last  latitude = {float(lats_original.isel(latitude=-1).values)}")

    lat_target = np.linspace(-90.0, 90.0, 73)

    try:
        var_interp = var.interp(latitude=lat_target)
    except Exception:
        var_interp = var.sortby("latitude").interp(latitude=lat_target)

    lats = xr.DataArray(
        lat_target,
        coords={"latitude": lat_target},
        dims=["latitude"],
        name="latitude"
    )

    print_da_info("var_interp_B", var_interp)

    var_anomalies = var_interp.groupby("time.dayofyear") - var_interp.groupby("time.dayofyear").mean("time")
    print_da_info("var_anomalies_B before NH slice", var_anomalies)

    var_anomalies = var_anomalies.sel(latitude=slice(20, 90))
    lats = lats.sel(latitude=slice(20, 90))

    print_da_info("var_anomalies_B 20-90N", var_anomalies)
    print_da_info("lats_B 20-90N", lats)

    lats_np = np.array(lats)

    # Modified weight formula
    coslat = np.cos(np.deg2rad(lats_np)).clip(0.0, 1.0)
    wgts_mod = np.sqrt(coslat)

    print("Modified Code-B weight diagnostics:")
    print(f"  lats_np shape = {lats_np.shape}")
    print(f"  coslat min/max = {np.nanmin(coslat):.6f} / {np.nanmax(coslat):.6f}")
    print(f"  weights min/max = {np.nanmin(wgts_mod):.6f} / {np.nanmax(wgts_mod):.6f}")
    print(f"  weight shape = {wgts_mod.shape}")

    gh_layer = np.array(var_anomalies)

    if len(np.shape(gh_layer)) == 3:
        gh_layer = np.reshape(gh_layer, (np.shape(gh_layer)[0], len(lats_np)))

    print("EOF input diagnostics (Modified Code B):")
    print(f"  gh_layer shape = {gh_layer.shape}")
    print(f"  weight shape   = {wgts_mod.shape}")

    solver = StandardEof(gh_layer, weights=wgts_mod, center=True)

    # EOF1 explained variance fraction
    varfrac_B = float(np.squeeze(solver.varianceFraction(neigs=1)))
    print(f"Modified Code B EOF1 explained variance fraction = {varfrac_B:.6f} ({varfrac_B*100:.2f}%)")

    AO_x = np.reshape(
        solver.pcs(npcs=1, pcscaling=1),
        (np.shape(gh_layer)[0],)
    )

    ao_B = xr.DataArray(
        AO_x,
        coords=[time],
        dims=["time"],
        name="ao_B_modw"
    )
    ao_B = clean_time_series_da(ao_B, "ao_B_modw")

    eof1_reg_B_lat = solver.eofsAsCovariance(neofs=1, pcscaling=1).squeeze()
    eof1_reg_B_lat = xr.DataArray(
        eof1_reg_B_lat,
        coords={"latitude": lats.values},
        dims=["latitude"],
        name="eof1_reg_B_modw_lat"
    )

    print_da_info("ao_B before sign check", ao_B)
    print_da_info("eof1_reg_B_lat before sign check", eof1_reg_B_lat)

    nof_lats = len(lats_np)
    max_AO = int(ao_B.argmax().values)

    print(f"max_AO index = {max_AO}")
    print(f"max_AO time  = {pd.to_datetime(ao_B.time.values[max_AO])}")

    if lats_orientation == "negative":
        left_mean = np.nanmean(gh_layer[max_AO, 0:int(nof_lats/2)-1])
        right_mean = np.nanmean(gh_layer[max_AO, int(nof_lats/2)+1:nof_lats-1])

        print("Using sign branch: lats_orientation == 'negative'")
        print(f"left_mean  = {left_mean:.6f}")
        print(f"right_mean = {right_mean:.6f}")
        print(f"left-right = {left_mean - right_mean:.6f}")

        if (left_mean - right_mean) < 0:
            flip_B = -1
            print("Modified Code B flipped (negative branch)")
        else:
            flip_B = 1
            print("Modified Code B kept sign (negative branch)")
    else:
        left_mean = np.nanmean(gh_layer[max_AO, 0:int(nof_lats/2)-3])
        right_mean = np.nanmean(gh_layer[max_AO, int(nof_lats/2)+3:nof_lats-1])

        print("Using sign branch: lats_orientation != 'negative'")
        print(f"left_mean  = {left_mean:.6f}")
        print(f"right_mean = {right_mean:.6f}")
        print(f"left-right = {left_mean - right_mean:.6f}")

        if (left_mean - right_mean) < 0:
            flip_B = -1
            print("Modified Code B flipped (positive branch)")
        else:
            flip_B = 1
            print("Modified Code B kept sign (positive branch)")

    ao_B = ao_B * flip_B
    eof1_reg_B_lat = eof1_reg_B_lat * flip_B

    eof1_reg_B_ring = expand_lat_pattern_to_2d(
        eof1_reg_B_lat,
        lon_template,
        name="eof1_reg_B_modw_ring"
    )

    print_da_info("ao_B final", ao_B)
    print_da_info("eof1_reg_B_lat final", eof1_reg_B_lat)
    print_da_info("eof1_reg_B_ring final", eof1_reg_B_ring)

    return ao_B, eof1_reg_B_lat, eof1_reg_B_ring, varfrac_B
# ============================================================
# 5) PLOT FULL-PERIOD SCATTER
# ============================================================
def plot_full_period_scatter(df_merge, out_png):
    print_header("7) PLOT FULL-PERIOD SCATTER")

    r_val, p_val = pearsonr(df_merge["ao_A"], df_merge["ao_B_modw"])
    slope, intercept = np.polyfit(df_merge["ao_A"], df_merge["ao_B_modw"], 1)

    print(f"Pearson r = {r_val:.6f}")
    print(f"p-value   = {p_val:.6e}")
    print(f"Linear fit: ao_B_modw = {slope:.4f} * ao_A + {intercept:.4f}")

    fig, ax = plt.subplots(figsize=(8, 8))

    ax.scatter(
        df_merge["ao_A"],
        df_merge["ao_B_modw"],
        s=18,
        alpha=0.55,
        edgecolors="k",
        linewidths=0.3
    )

    xy_min = min(df_merge["ao_A"].min(), df_merge["ao_B_modw"].min())
    xy_max = max(df_merge["ao_A"].max(), df_merge["ao_B_modw"].max())
    pad = 0.08 * (xy_max - xy_min)
    xy_min -= pad
    xy_max += pad

    ax.plot([xy_min, xy_max], [xy_min, xy_max], "k--", lw=1.5, label="1:1 line")

    xfit = np.linspace(xy_min, xy_max, 200)
    yfit = slope * xfit + intercept
    ax.plot(xfit, yfit, color="red", lw=2, label=f"Fit: y={slope:.2f}x+{intercept:.2f}")

    ax.set_xlim(xy_min, xy_max)
    ax.set_ylim(xy_min, xy_max)
    ax.set_aspect("equal", adjustable="box")

    ax.set_xlabel("AO from Code A", fontsize=13)
    ax.set_ylabel("AO from modified-weight Code-B", fontsize=13)
    ax.set_title(
        f"AO Comparison on Identical Dates\n"
        f"Pearson r = {r_val:.3f}, p = {p_val:.2e}, N = {len(df_merge)}",
        fontsize=14,
        fontweight="bold"
    )
    ax.grid(True, alpha=0.3)
    ax.legend()

    plt.tight_layout()
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    print(f"Scatter plot saved to: {out_png}")

# ============================================================
# 6) PLOT SHORT-WINDOW TIMESERIES WITH NOAA
# ============================================================
def plot_short_window_timeseries(df_merge, df_cpc, out_png):
    print_header("8) PLOT SHORT-WINDOW TIMESERIES WITH NOAA")

    df_plot_ab = df_merge[
        (df_merge["time"] >= pd.to_datetime(PLOT_START)) &
        (df_merge["time"] <= pd.to_datetime(PLOT_END))
    ].copy()

    df_plot_cpc = df_cpc[
        (df_cpc["time"] >= pd.to_datetime(PLOT_START)) &
        (df_cpc["time"] <= pd.to_datetime(PLOT_END))
    ].copy()

    df_plot = pd.merge(df_plot_ab, df_plot_cpc, on="time", how="inner").dropna().sort_values("time").reset_index(drop=True)

    print(f"PLOT_START = {PLOT_START}")
    print(f"PLOT_END   = {PLOT_END}")
    print(f"AB plot rows   = {len(df_plot_ab)}")
    print(f"CPC plot rows  = {len(df_plot_cpc)}")
    print(f"Final plot rows= {len(df_plot)}")

    if len(df_plot) == 0:
        raise ValueError(f"No matched A/B/CPC dates found between {PLOT_START} and {PLOT_END}.")

    r_A_CPC, p_A_CPC = pearsonr(df_plot["ao_A"], df_plot["ao_cpc"])
    r_B_CPC, p_B_CPC = pearsonr(df_plot["ao_B_modw"], df_plot["ao_cpc"])
    r_A_B, p_A_B = pearsonr(df_plot["ao_A"], df_plot["ao_B_modw"])

    print("Window correlations:")
    print(f"  A vs CPC    : r = {r_A_CPC:.4f}, p = {p_A_CPC:.3e}")
    print(f"  Bmodw vs CPC: r = {r_B_CPC:.4f}, p = {p_B_CPC:.3e}")
    print(f"  A vs Bmodw  : r = {r_A_B:.4f}, p = {p_A_B:.3e}")

    fig, ax = plt.subplots(figsize=(12, 8))

    ax.plot(df_plot["time"], df_plot["ao_cpc"], lw=2.6, color="black", label="NOAA CPC Official")
    ax.plot(df_plot["time"], df_plot["ao_A"], lw=1.8, color="red", linestyle="--", label="Code A")
    ax.plot(df_plot["time"], df_plot["ao_B_modw"], lw=1.8, color="blue", linestyle="-.", label="Modified-weight Code B")

    ax.axhline(0, color="gray", lw=1.0)
    ax.set_ylabel("Standardized AO Index", fontsize=14)
    ax.set_xlabel("Time", fontsize=14)
    ax.set_title(
        f"AO Comparison ({PLOT_START} to {PLOT_END})\n"
        f"A vs CPC r={r_A_CPC:.3f}, Bmodw vs CPC r={r_B_CPC:.3f}, A vs Bmodw r={r_A_B:.3f}",
        fontsize=15,
        fontweight="bold"
    )
    ax.legend(loc="upper right", fontsize=12)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    print(f"Short-window timeseries plot saved to: {out_png}")

# ============================================================
# 7) PLOT SPATIAL PATTERN COMPARISON
# ============================================================
def plot_two_polar_maps(mapA, mapB, varfrac_A, varfrac_B, out_png):
    print_header("9) PLOT SPATIAL PATTERN COMPARISON")

    mapA_plot = mapA.sortby("latitude")
    mapB_plot = mapB.sortby("latitude")

    noaa_colors = [
        "#2d1e8f",
        "#4355c7",
        "#5a8ce6",
        "#7ebbf5",
        "#a3dcfb",
        "#c2f0fb",
        "#dff8fd",
        "#f0fcfd",
        "#ffffff",
        "#fefbd9",
        "#feea9e",
        "#fec762",
        "#f27932",
        "#b82522"
    ]
    bounds = [-45, -40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25, 30]
    cbar_ticks = [-40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25]

    cmap_noaa = mcolors.ListedColormap(noaa_colors)
    norm_noaa = mcolors.BoundaryNorm(bounds, cmap_noaa.N)

    fig, axes = plt.subplots(
        1, 2, figsize=(14, 7),
        subplot_kw={"projection": ccrs.NorthPolarStereo()}
    )

    for ax, da, ttl in zip(
        axes,
        [mapA_plot, mapB_plot],
        [
            f"Code A: 2D EOF regression/covariance pattern\nEOF1 explained variance = {varfrac_A*100:.2f}%",
            f"Modified-weight Code B: zonal-mean EOF expanded to 2D ring pattern\nEOF1 explained variance = {varfrac_B*100:.2f}%"
        ]
    ):
        ax.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())
        ax.coastlines(linewidth=1)
        ax.add_feature(cfeature.BORDERS, linestyle=":", alpha=0.5)

        lon = da.longitude.values
        lat = da.latitude.values
        field_cyc, lon_cyc = add_cyclic_point(da.values, coord=lon)

        cf = ax.contourf(
            lon_cyc, lat, field_cyc,
            levels=bounds,
            transform=ccrs.PlateCarree(),
            cmap=cmap_noaa,
            norm=norm_noaa,
            extend="both"
        )

        ax.set_title(ttl, fontsize=12, fontweight="bold", pad=14)

    fig.suptitle(
        "AO Spatial Patterns: Code A vs Modified-weight Code-B Ring Pattern",
        fontsize=15,
        fontweight="bold",
        y=0.95
    )

    # 按你给的参考版本调整版面
    fig.subplots_adjust(left=0.05, right=0.95, top=0.86, bottom=0.18, wspace=0.14)

    # 独立 colorbar 轴
    cbar_ax = fig.add_axes([0.20, 0.08, 0.60, 0.035])
    cbar = fig.colorbar(
        cf,
        cax=cbar_ax,
        orientation="horizontal",
        ticks=cbar_ticks
    )
    cbar.set_label("Regression / covariance map of 1000 hPa height (m)", fontsize=12)

    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    print(f"Spatial pattern comparison map saved to: {out_png}")

# ============================================================
# 8) PLOT LATITUDE PROFILE COMPARISON
# ============================================================
def plot_lat_profiles(mapA, lat_pattern_B, out_png):
    print_header("10) PLOT LATITUDE PROFILE COMPARISON")

    A_zm = mapA.mean("longitude").sortby("latitude")
    B_lat = lat_pattern_B.sortby("latitude")

    B_on_A = B_lat.interp(latitude=A_zm.latitude)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(A_zm.latitude, A_zm, lw=2.2, color="red", label="Code A zonal-mean profile")
    ax.plot(B_on_A.latitude, B_on_A, lw=2.2, color="blue", linestyle="--", label="Modified-weight Code B 1D lat pattern")

    ax.axhline(0, color="gray", lw=1.0)
    ax.set_xlim(20, 90)
    ax.set_xlabel("Latitude (°N)", fontsize=12)
    ax.set_ylabel("Regression / covariance height anomaly (m)", fontsize=12)
    ax.set_title("AO spatial pattern latitude profile comparison", fontsize=14, fontweight="bold")
    ax.grid(True, alpha=0.3)
    ax.legend()

    plt.tight_layout()
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    print(f"Latitude-profile comparison plot saved to: {out_png}")

# ============================================================
# 9) MAIN
# ============================================================
def main():
    Z_daily_global = open_era5_daily_z1000_global()

    ao_A_da, eof1_reg_A, Z_daily_A, varfrac_A = compute_codeA_ao_and_pattern(Z_daily_global)
    ao_A_da = maybe_filter_time_da(ao_A_da, COMPARE_START, COMPARE_END)
    ao_A_da.to_netcdf(OUT_A_AO_NC)
    print(f"AO_A saved to: {OUT_A_AO_NC}")

    eof1_reg_A.to_netcdf(OUT_A_MAP_NC)
    print(f"Code A spatial pattern saved to: {OUT_A_MAP_NC}")

    ao_B_da, eof1_reg_B_lat, eof1_reg_B_ring, varfrac_B = compute_codeB_modified_weights_ao_and_pattern(
        Z_daily_global,
        lon_template=Z_daily_A.longitude
    )
    ao_B_da = maybe_filter_time_da(ao_B_da, COMPARE_START, COMPARE_END)
    ao_B_da.to_netcdf(OUT_B_AO_NC)
    print(f"AO_B_modw saved to: {OUT_B_AO_NC}")

    eof1_reg_B_lat.to_netcdf(OUT_B_LAT_NC)
    print(f"Modified-weight Code B latitude pattern saved to: {OUT_B_LAT_NC}")

    eof1_reg_B_ring.to_netcdf(OUT_B_RING_NC)
    print(f"Modified-weight Code B ring pattern saved to: {OUT_B_RING_NC}")

    df_A = pd.DataFrame({
        "time": pd.to_datetime(ao_A_da.time.values),
        "ao_A": ao_A_da.values
    }).dropna().sort_values("time").reset_index(drop=True)

    df_B = pd.DataFrame({
        "time": pd.to_datetime(ao_B_da.time.values),
        "ao_B_modw": ao_B_da.values
    }).dropna().sort_values("time").reset_index(drop=True)

    print_header("5) MERGE ON IDENTICAL DATES")
    df_merge = pd.merge(df_A, df_B, on="time", how="inner").dropna().sort_values("time").reset_index(drop=True)

    if len(df_merge) == 0:
        raise ValueError("AO_A and AO_B_modw have no common dates.")

    print(df_merge.head())
    print(f"Merged length = {len(df_merge)}")
    print(f"Merged time range = {df_merge['time'].min()} to {df_merge['time'].max()}")

    df_merge.to_csv(OUT_MERGEDCSV, index=False)
    print(f"Merged CSV saved to: {OUT_MERGEDCSV}")

    plot_full_period_scatter(df_merge, OUT_SCATTER)

    df_cpc = load_noaa_ao_txt(AO_FILE)
    plot_short_window_timeseries(df_merge, df_cpc, OUT_TS)

    plot_two_polar_maps(eof1_reg_A, eof1_reg_B_ring, varfrac_A, varfrac_B, OUT_MAP_COMPARE)
    plot_lat_profiles(eof1_reg_A, eof1_reg_B_lat, OUT_LATPROFILE)

    print("\nAll tasks completed.")

if __name__ == "__main__":
    main()

In [ ]:
import os
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

OUT_DIR = "/home/weiji/restart_exam/code/20260331_AO_compare_A_vs_Bmodw"

A_MAP_FILE = os.path.join(OUT_DIR, "AO_codeA_regmap.nc")
B_MAP_FILE = os.path.join(OUT_DIR, "AO_codeB_modified_weights_ring_regmap.nc")
OUT_PNG = os.path.join(OUT_DIR, "AO_spatial_patterns_A_vs_Bmodw_ring_REPLOT.png")


def load_first_var(ncfile):
    ds = xr.open_dataset(ncfile)
    varname = list(ds.data_vars)[0]
    return ds[varname]


def replot_two_polar_maps(mapA, mapB, out_png):
    mapA_plot = mapA.sortby("latitude")
    mapB_plot = mapB.sortby("latitude")

    noaa_colors = [
        "#2d1e8f",
        "#4355c7",
        "#5a8ce6",
        "#7ebbf5",
        "#a3dcfb",
        "#c2f0fb",
        "#dff8fd",
        "#f0fcfd",
        "#ffffff",
        "#fefbd9",
        "#feea9e",
        "#fec762",
        "#f27932",
        "#b82522"
    ]
    bounds = [-45, -40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25, 30]
    cbar_ticks = [-40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25]

    cmap_noaa = mcolors.ListedColormap(noaa_colors)
    norm_noaa = mcolors.BoundaryNorm(bounds, cmap_noaa.N)

    fig, axes = plt.subplots(
        1, 2, figsize=(14, 7),
        subplot_kw={"projection": ccrs.NorthPolarStereo()}
    )

    for ax, da, ttl in zip(
        axes,
        [mapA_plot, mapB_plot],
        [
            "Code A: 2D EOF regression/covariance pattern",
            "Modified-weight Code B: zonal-mean EOF expanded to 2D ring pattern"
        ]
    ):
        ax.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())
        ax.coastlines(linewidth=1)
        ax.add_feature(cfeature.BORDERS, linestyle=":", alpha=0.5)

        lon = da.longitude.values
        lat = da.latitude.values
        field_cyc, lon_cyc = add_cyclic_point(da.values, coord=lon)

        cf = ax.contourf(
            lon_cyc, lat, field_cyc,
            levels=bounds,
            transform=ccrs.PlateCarree(),
            cmap=cmap_noaa,
            norm=norm_noaa,
            extend="both"
        )

        ax.set_title(ttl, fontsize=12, fontweight="bold", pad=14)

    fig.suptitle(
        "AO Spatial Patterns: Code A vs Modified-weight Code-B Ring Pattern",
        fontsize=15,
        fontweight="bold",
        y=0.95
    )

    # manually place the two map panels higher, leaving room for colorbar
    fig.subplots_adjust(left=0.05, right=0.95, top=0.86, bottom=0.18, wspace=0.14)

    # independent colorbar axis below the plots
    cbar_ax = fig.add_axes([0.20, 0.08, 0.60, 0.035])
    cbar = fig.colorbar(
        cf,
        cax=cbar_ax,
        orientation="horizontal",
        ticks=cbar_ticks
    )
    cbar.set_label("Regression / covariance map of 1000 hPa height (m)", fontsize=12)

    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved to: {out_png}")


mapA = load_first_var(A_MAP_FILE)
mapB = load_first_var(B_MAP_FILE)

replot_two_polar_maps(mapA, mapB, OUT_PNG)